# Part별 공정조합 평균 — 공정이력 유사성과 공유 웨이퍼 보정

기존 `real_data_funnel_by_part.ipynb`와 **같은 실제 `raw.csv` 입력**을 사용합니다.
제품을 섞지 않으며, 원본 후보 생성·부모 gate·G/B 제외·조합의 웨이퍼 수·평균은 그대로 유지합니다.
기본 변환도 최근 버전과 같은 `rank_normal`입니다. `Y_TRANSFORM='raw'`면 원래 y,
`'yeo_johnson'`이면 기존 강건 Yeo–Johnson을 사용합니다. 원본 raw 평균은 항상 별도 보존합니다.

## 새 보정의 의미

1. **이력 표현:** 같은 STEP·Order에서 사용한 장비·챔버를 비교합니다. 결과 y나 선택된 후보의
   반복 등장 횟수로 유사도를 만들지 않습니다. 동일 part의 전체 준비된 이력을 사용하고,
   관측 장비 종류가 2개 미만이거나 coverage가 기준보다 작은 위치는 제외합니다.
   남은 위치는 고정된 같은 가중치를 가지며, 결측 둘을 같은 장비로 취급하지 않습니다.
   이력은 기존 `REWORK='first'/'last'` 규칙으로 정리한 것이므로 재작업 횟수 자체는 모델링하지 않습니다.
2. **연속적인 공유 영향:** 가중 Hamming 거리 $d_{ij}$에 대해
   $K_{ij}=\exp(-d_{ij}/\ell)$, $\Sigma=\sigma^2[(1-\rho)I+\rho K]$를 작업 모형으로 사용합니다.
   완전히 같은 이력이 없어도 적용됩니다. 길이 척도 $\ell$, 공유 비율 $\rho$, 분산을
   제한된 수의 웨이퍼에서 Gaussian REML로 선택·추정합니다. 유사도는 측정된 상관계수가 아닙니다.
   비교 대상으로 iid 모형($\rho=0$)도 포함하며, 근거가 없으면 억지로 상관을 만들지 않습니다.
3. **큰 데이터:** 전체 웨이퍼×웨이퍼 행렬 대신 제한된 대표 이력의 Nyström 저차원 근사와
   대각 잔차를 사용합니다. 대표 수, 적합 표본 수, MC 연산량을 설정으로 제한합니다.
   근사 오차와 잔차 비중을 출력합니다. 대각 잔차 근사는 anchor로 표현되지 않은
   동일/유사 이력의 상관을 덜 반영할 수 있습니다. 필요하면 대표 수를 늘려 민감도를 확인하세요.
4. **전체 평균과 비교:** $a_g$는 조합 내 웨이퍼에 $1/n_g$를 주는 벡터이고
   $p=(1/N,\ldots,1/N)$이면, $SE_g^2=(a_g-p)^\top\Sigma(a_g-p)$를 계산합니다.
   전체 part에 공통인 영향은 상쇄됩니다. 같은 n이어도 구성에 따라 SE가 달라집니다.
   `history_n_eff`는 조합 평균 자체에 대한 정보량 진단이며, 실제 판정에는 위 contrast SE를 씁니다.
5. **기대 극값:** $Z_g=(\bar y_g-\bar y_{part})/SE_g$와
   $c_n=E[\max_{g:n_g=n}|Z_g|]$를 사용합니다. 동일 n의 실제 타점 수 $m_n$을 유지합니다.
   모형에서 웨이퍼 단위 공동 Gaussian draw를 생성하고 **최대 절대편차를 평균**내므로,
   이력 상관뿐 아니라 타점 사이의 웨이퍼 재사용과 전체 평균 공유도 반영합니다.
   데이터의 경험적 분위수를 쓰지 않습니다. MCSE는 이 수치적 기대값 계산의 오차입니다.

왼쪽은 기존 평균 차트, 오른쪽은 보정 SE로 환산한 차트입니다. 보정 경계는 원래 분석값 단위로도
`history_lower/upper = part 평균 ± SE_g × c_n`에 저장합니다. 기존 κ는 보정 모형에 다시 곱하지 않습니다.
모든 산점도는 같은 작은 점이며, 기존만 밖은 파랑, 이력 보정만 밖은 초록, 양쪽 밖은 빨강입니다.

**해석:** 기대 극값은 신뢰구간·유의수준·오탐률 보장선이 아닙니다. 적합된 모형을 고정한 탐색적 비교입니다.
실제 공정 효과가 공분산 적합에 흡수될 수도 있으며, raw y 기반 부모 gate로 선별한 후보를 고정해
시뮬레이션하므로 후보 선택 효과나 모형 추정 불확실성까지 교정하지 않습니다.
보정 후 경계 안으로 들어왔다는 이유만으로 정상 공정이라고 결론 내리지 마세요.
`HISTORY_CORRECTION=False`는 이력 공유 성분을 끄되, 오른쪽에서 공유 웨이퍼·전체 평균 효과는 유지합니다.

필요 패키지: `numpy pandas polars scipy matplotlib`.
미설치 시 별도 셀에서 `%pip install numpy pandas polars scipy matplotlib` 실행 후 커널을 재시작하세요.

참고: [공분산 함수와 커널](https://gaussianprocess.org/gpml/chapters/RW4.pdf),
[혼합 모형의 REML](https://www.jstatsoft.org/article/view/v067i01).


In [ ]:
from pathlib import Path

# notebooks 폴더에서 실행하면 저장소 루트의 raw.csv를 읽습니다.
DATA_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CSV_PATH = DATA_DIR / 'raw.csv'
# CSV_PATH = Path('/absolute/path/to/raw.csv')

Y_TRANSFORM = 'rank_normal'  # 'rank_normal': 순위 정규변환 / 'yeo_johnson': YJ / 'raw': 변환 OFF
# 변환 옵션을 바꾸면 Run All. YJ_* 설정은 Y_TRANSFORM='yeo_johnson'일 때만 사용합니다.
YJ_LAMBDA_METHOD = 'robust'   # 'robust': RewML 기반 강건 추정 / 'mle': 이전 전체 데이터 MLE
YJ_ROBUST_FALLBACK = 'mle'    # 강건 추정 불가: 'mle'로 명시 전환 / 'error'이면 해당 part 제외

EXTREME_COEFFICIENT = 'absolute_max'  # 'absolute_max': 유한 m의 E[max|Z|] / 'leading': 이전 √(2 ln m)
SCATTER_SIZE = 5.0          # 모든 산점도의 동일한 작은 점 크기 (면적, pt²)

EMPIRICAL_ENVELOPE = True   # 기존 κ 결과를 표에 보존. 이력 보정에는 다시 곱하지 않음
EXTREME_FIT_MIN_GROUPS = 5   # 적합에 필요한 서로 다른 n 구간 수 (각 m_n >= 2)
EXTREME_FIT_HUBER_DELTA = 1.5 # 표준화 극값 잔차의 Huber 전환점

PART_IDS = None              # None: 모든 part. 예: ['PRODUCT_A', 'PRODUCT_B']
LINE_ID = None               # 특정 line만 사용할 때 지정
STEP = None                  # None: 모든 STEP. 예: 'ex102000'
MIN_N = 20                   # 실제 조합의 최소 고유 웨이퍼 수
MAX_DEPTH = 3                # 한 조합에 묶을 Order 수: 1 ~ MAX_DEPTH
STEP_SUFFIX = r'_(\d+)$'     # step_seq 끝의 _숫자를 Order로 해석. None이면 step_ord 사용
REWORK = 'last'              # 같은 wafer·STEP·Order 재작업: 'first' 또는 'last'
LOG_X = True
OUTSIDE_ROWS = 30            # 표에 표시할 경계 밖 조합 수. 차트에는 모든 점 표시
SAVE_DIR = None              # 예: DATA_DIR / 'part_funnel_results'
HIGHER_IS_WORSE = True       # raw 후보 선별 방향. 표시값의 부호는 뒤집지 않음
SHOW_PROGRESS = True        # part 시작과 긴 계산의 진행 상황 표시

# 이력 유사성 보정: 같은 입력으로 새 분석을 추가합니다.
HISTORY_CORRECTION = True
HISTORY_MIN_COVERAGE = 0.5
HISTORY_MAX_LANDMARKS = 128  # 정확도/시간 절충. 근사 오차가 크면 256 이상으로 비교
HISTORY_FIT_WAFERS = 1200   # 공분산 적합에 사용할 최대 웨이퍼 수
HISTORY_MC_DRAWS = 1024     # 모형에서 최대 절대편차의 기대값을 계산하는 draw 수
HISTORY_MC_OPERATION_BUDGET = 250_000_000  # 소속 행렬+저차원 곱의 추정 연산량. 최소 128회가 우선
HISTORY_SEED = 42
KEEP_HISTORY_MODEL = False  # True: 재시뮬레이션용 N×rank 공분산 특징도 report에 보존
SHOW_LEGACY_SCALE_FIT = False  # True: 왼쪽에 기존 κ 경계를 참고로 추가
SHOW_IID_COEFFICIENT = False   # True: 오른쪽에 독립 타점 기대 극값을 참고로 추가


In [ ]:
import hashlib
import html
import math
import time
import re
from itertools import combinations

import numpy as np
import pandas as pd
try:
    import polars as pl
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError('Polars가 필요합니다. Jupyter 셀에서 %pip install "polars>=1.0" 실행 후 커널을 재시작하세요.') from exc
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import HTML, display

assert isinstance(MIN_N, int) and not isinstance(MIN_N, bool) and MIN_N >= 1
assert isinstance(MAX_DEPTH, int) and not isinstance(MAX_DEPTH, bool) and MAX_DEPTH >= 1
assert REWORK in ('first', 'last')
assert Y_TRANSFORM in ('rank_normal', 'yeo_johnson', 'raw')
assert YJ_LAMBDA_METHOD in ('robust', 'mle')
assert YJ_ROBUST_FALLBACK in ('mle', 'error')
assert EXTREME_COEFFICIENT in ('absolute_max', 'leading')
assert np.isfinite(SCATTER_SIZE) and SCATTER_SIZE > 0
assert isinstance(EMPIRICAL_ENVELOPE, (bool, np.bool_))
assert isinstance(EXTREME_FIT_MIN_GROUPS, int) and not isinstance(EXTREME_FIT_MIN_GROUPS, bool) and EXTREME_FIT_MIN_GROUPS >= 2
assert np.isfinite(EXTREME_FIT_HUBER_DELTA) and EXTREME_FIT_HUBER_DELTA > 0
assert isinstance(OUTSIDE_ROWS, int) and OUTSIDE_ROWS >= 1

font_dirs = [Path.cwd() / 'fonts', Path.cwd() / 'notebooks' / 'fonts']
font_file = next((d / 'NanumGothic-Regular.ttf' for d in font_dirs if (d / 'NanumGothic-Regular.ttf').is_file()), None)
if font_file is not None:
    mpl.font_manager.fontManager.addfont(str(font_file))
    font_name = mpl.font_manager.FontProperties(fname=str(font_file)).get_name()
else:
    available = {f.name for f in mpl.font_manager.fontManager.ttflist}
    font_name = next((name for name in ['NanumGothic', 'Noto Sans CJK KR', 'Malgun Gothic', 'AppleGothic'] if name in available), 'DejaVu Sans')
plt.rcParams.update({'font.family': font_name, 'axes.unicode_minus': False, 'figure.dpi': 120})
pd.set_option('display.max_columns', 20)
pd.set_option('display.precision', 8)

WAFER_KEY = ['root_lot_id', 'wafer_id']
REQUIRED = ['part_id', 'root_lot_id', 'wafer_id', 'y_value', 'step_seq', 'eqp_id']
OPTIONAL = ['chamber_id', 'step_ord', 'tkin_time', 'line_id', 'good_bad']
MISSING = 0xFFFF

def part_heading(part_id):
    display(HTML('<h3>part_id = ' + html.escape(str(part_id)) + '</h3>'))

try:
    import scipy
    from scipy import sparse
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError('이력 보정에는 scipy가 필요합니다. %pip install scipy 후 다시 실행하세요.') from exc
assert isinstance(HISTORY_CORRECTION, (bool, np.bool_))
assert 0 <= HISTORY_MIN_COVERAGE <= 1
assert isinstance(HISTORY_MAX_LANDMARKS, int) and HISTORY_MAX_LANDMARKS >= 2
assert isinstance(HISTORY_FIT_WAFERS, int) and HISTORY_FIT_WAFERS >= 4
assert isinstance(HISTORY_MC_DRAWS, int) and HISTORY_MC_DRAWS >= 128
assert HISTORY_MC_OPERATION_BUDGET > 0


## 1. 실제 CSV 읽기 — Polars
필요한 열만 읽고, part별 웨이퍼와 공정 이력을 준비합니다. 같은 웨이퍼의 y가 여러 개면 원본처럼 첫 유효 값을 사용하고 건수를 알립니다.
원본과 같이 G/B 값이 존재하면 해당 웨이퍼만 분석합니다. G/B가 전혀 없으면 raw y만으로 분석합니다.


In [ ]:
def read_raw_csv(path):
    path = Path(path).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f'실제 raw.csv를 찾을 수 없습니다: {path}\nCSV_PATH를 지정하세요.')
    columns = pl.read_csv(path, n_rows=0, infer_schema=False).columns
    missing = sorted(set(REQUIRED) - set(columns))
    if missing:
        raise ValueError(f'CSV에 필요한 컬럼이 없습니다: {missing}')
    if LINE_ID is not None and 'line_id' not in columns:
        raise ValueError('LINE_ID를 지정했지만 CSV에 line_id가 없습니다.')
    use = REQUIRED + [c for c in OPTIONAL if c in columns]
    # pandas 기본 결측 표기와 동일하게 읽는다. 식별자의 앞자리 0은 문자열로 보존한다.
    null_values = ['', '#N/A', '#N/A N/A', '#NA', '-1.#IND', '-1.#QNAN', '-NaN', '-nan',
                   '1.#IND', '1.#QNAN', '<NA>', 'N/A', 'NA', 'NULL', 'NaN', 'None', 'n/a', 'nan', 'null']
    raw = pl.read_csv(path, columns=use, infer_schema=False, null_values=null_values)
    rows_read = raw.height
    raw = raw.with_columns([pl.col(c).str.strip_chars().replace('', None).alias(c) for c in use])
    if LINE_ID is not None:
        raw = raw.filter(pl.col('line_id') == str(LINE_ID))
    missing_key = raw.select(pl.any_horizontal(pl.col(['part_id'] + WAFER_KEY).is_null()).sum()).item()
    raw = raw.drop_nulls(['part_id'] + WAFER_KEY)
    found_parts = sorted(raw['part_id'].unique().to_list())
    selected = found_parts if PART_IDS is None else ([str(PART_IDS)] if isinstance(PART_IDS, (str, int)) else list(dict.fromkeys(map(str, PART_IDS))))
    unknown = sorted(set(selected) - set(found_parts))
    if unknown:
        raise ValueError(f'CSV에 없는 part_id: {unknown}')
    if not selected:
        raise ValueError('분석할 part가 없습니다.')
    raw = raw.filter(pl.col('part_id').is_in(selected)).with_row_index('_row')
    wid_number = pl.col('wafer_id').cast(pl.Float64, strict=False)
    raw = raw.with_columns(
        pl.when(wid_number.is_finite() & (wid_number == wid_number.floor()))
          .then(wid_number.cast(pl.Int64, strict=False).cast(pl.String))
          .otherwise(pl.col('wafer_id')).alias('wafer_id'),
        pl.col('y_value').cast(pl.Float64, strict=False).alias('_y'),
    ).with_columns(pl.when(pl.col('_y').is_finite()).then(pl.col('_y')).otherwise(None).alias('_y'))
    if 'chamber_id' not in raw.columns:
        raw = raw.with_columns(pl.lit('').alias('chamber_id'))
    else:
        raw = raw.with_columns(pl.col('chamber_id').fill_null(''))
    if 'good_bad' in raw.columns:
        raw = raw.with_columns(pl.col('good_bad').str.to_uppercase().alias('_gb'))
        raw = raw.with_columns(pl.when(pl.col('_gb') == 'G').then(0.0)
                              .when(pl.col('_gb') == 'B').then(1.0).otherwise(None).alias('_bad'))
    else:
        raw = raw.with_columns(pl.lit(None, dtype=pl.String).alias('_gb'), pl.lit(None, dtype=pl.Float64).alias('_bad'))
    # 시각 해석은 원본과 같은 pandas 규칙을 사용하고, 정렬·집계는 Polars에서 수행한다.
    if 'tkin_time' in raw.columns:
        times = pd.to_datetime(raw['tkin_time'].to_numpy(), errors='coerce', utc=True)
        raw = raw.with_columns(pl.Series('_time', times.to_numpy(dtype='datetime64[ns]')))
    else:
        raw = raw.with_columns(pl.lit(None, dtype=pl.Datetime('ns')).alias('_time'))
    fallback = pl.col('step_ord').cast(pl.Float64, strict=False) if 'step_ord' in raw.columns else pl.lit(None, dtype=pl.Float64)
    if STEP_SUFFIX:
        raw = raw.with_columns(pl.col('step_seq').str.replace_all(STEP_SUFFIX, '').alias('_step'),
                               pl.coalesce(pl.col('step_seq').str.extract(STEP_SUFFIX, 1).cast(pl.Float64, strict=False), fallback).alias('_order'))
    else:
        raw = raw.with_columns(pl.col('step_seq').alias('_step'), fallback.alias('_order'))
    print(f'실제 데이터: {path}\nPolars {pl.__version__} · 읽은 행 {rows_read:,} · 식별자 결측 제외 {missing_key:,} · part {len(selected):,}개')
    return raw, selected


raw, selected_parts = read_raw_csv(CSV_PATH)
inventory = raw.group_by('part_id').len().sort('part_id')
display(pd.DataFrame(inventory.to_dict(as_series=False)).rename(columns={'len': '이력 행 수'}))


## 2. 원본과 같은 공정조합 후보 생성
- 각 STEP에서 유닛이 2개 이상인 Order만 사용합니다.
- 깊이 1~2는 MIN_N을 만족하는 관측 조합을 계산합니다.
- 깊이 3 이상은 원본과 같이 부모 조합의 z 관문을 통과한 후보만 계산합니다.
- STEP 설정은 원본처럼 표시 범위만 정합니다. 후보 생성과 부모 선별은 전체 STEP에서 먼저 수행합니다.

순위 정규변환 / Yeo–Johnson / raw 중 어느 것을 선택해도 부모 선별에는 기존 raw y를 사용하므로 동일 입력의 후보 수가 같습니다.
아래 수치 함수와 후보 생성 함수는 원본 구현에서 가져왔습니다. 작은 정수 키는 정렬 없이 집계하도록 최적화했습니다.


In [ ]:
Z95, FWER_ALPHA = 1.959964, 0.05
SQRT2 = math.sqrt(2)
_A = (-39.69683028665376, 220.9460984245205, -275.9285104469687, 138.357751867269, -30.66479806614716, 2.506628277459239)
_B = (-54.47609879822406, 161.5858368580409, -155.6989798598866, 66.80131188771972, -13.28068155288572)
_C = (-0.007784894002430293, -0.3223964580411365, -2.400758277161838, -2.549732539343734, 4.374664141464968, 2.938163982698783)
_D = (0.007784695709041462, 0.3224671290700398, 2.445134137142996, 3.754408661907416)
_LG = (76.18009172947146, -86.50532032941677, 24.01409824083091, -1.231739572450155, 0.1208650973866179e-2, -0.5395239384953e-5)


def erfc(x):
    x = np.asarray(x, dtype=float)
    z = np.abs(x)
    t = 1 / (1 + 0.5 * z)
    poly = 1.00002368 + t * (0.37409196 + t * (0.09678418 + t * (-0.18628806 + t * (0.27886807 + t * (-1.13520398 + t * (1.48851587 + t * (-0.82215223 + t * 0.17087277)))))))
    r = t * np.exp(-z * z - 1.26551223 + t * poly)
    return np.where(x >= 0, r, 2 - r)


def norm_cdf(z):
    return 0.5 * erfc(-np.asarray(z, dtype=float) / SQRT2)


def norm_sf(z):
    return 0.5 * erfc(np.asarray(z, dtype=float) / SQRT2)


def norm_inv(p):
    if p <= 0:
        return -math.inf
    if p >= 1:
        return math.inf
    if p < 0.02425:
        q = math.sqrt(-2 * math.log(p))
        x = (((((_C[0] * q + _C[1]) * q + _C[2]) * q + _C[3]) * q + _C[4]) * q + _C[5]) / ((((_D[0] * q + _D[1]) * q + _D[2]) * q + _D[3]) * q + 1)
    elif p <= 1 - 0.02425:
        q = p - 0.5
        r = q * q
        x = ((((((_A[0] * r + _A[1]) * r + _A[2]) * r + _A[3]) * r + _A[4]) * r + _A[5]) * q) / (((((_B[0] * r + _B[1]) * r + _B[2]) * r + _B[3]) * r + _B[4]) * r + 1)
    else:
        q = math.sqrt(-2 * math.log(1 - p))
        x = -(((((_C[0] * q + _C[1]) * q + _C[2]) * q + _C[3]) * q + _C[4]) * q + _C[5]) / ((((_D[0] * q + _D[1]) * q + _D[2]) * q + _D[3]) * q + 1)
    e = float(norm_cdf(x)) - p
    u = e * math.sqrt(2 * math.pi) * math.exp((x * x) / 2)
    return x - u / (1 + (x * u) / 2)


def norm_isf(p):
    """상단 꼬리확률 → z (배열, 아주 작은 p도 정밀하게)"""
    p = np.clip(np.atleast_1d(np.asarray(p, dtype=float)), 1e-300, 1 - 1e-16)
    x = np.empty_like(p)
    lo, hi = p < 0.02425, p > 1 - 0.02425
    mid = ~(lo | hi)
    q = np.sqrt(-2 * np.log(p[lo]))
    x[lo] = (((((_C[0] * q + _C[1]) * q + _C[2]) * q + _C[3]) * q + _C[4]) * q + _C[5]) / ((((_D[0] * q + _D[1]) * q + _D[2]) * q + _D[3]) * q + 1)
    q = p[mid] - 0.5
    r = q * q
    x[mid] = ((((((_A[0] * r + _A[1]) * r + _A[2]) * r + _A[3]) * r + _A[4]) * r + _A[5]) * q) / (((((_B[0] * r + _B[1]) * r + _B[2]) * r + _B[3]) * r + _B[4]) * r + 1)
    q = np.sqrt(-2 * np.log(1 - p[hi]))
    x[hi] = -(((((_C[0] * q + _C[1]) * q + _C[2]) * q + _C[3]) * q + _C[4]) * q + _C[5]) / ((((_D[0] * q + _D[1]) * q + _D[2]) * q + _D[3]) * q + 1)
    with np.errstate(all='ignore'):
        e = norm_cdf(x) - p
        u = e * math.sqrt(2 * math.pi) * np.exp((x * x) / 2)
        refined = x - u / (1 + (x * u) / 2)
    return -np.where(np.isfinite(refined) & (p > 1e-250), refined, x)


def lgamma(x):
    x = np.asarray(x, dtype=float)
    y = x.copy()
    tmp = x + 5.5 - (x + 0.5) * np.log(x + 5.5)
    ser = 1.000000000190015
    for g in _LG:
        y = y + 1
        ser = ser + g / y
    return -tmp + np.log((2.5066282746310005 * ser) / x)


def betacf(a, b, x):
    a, b, x = np.broadcast_arrays(*(np.asarray(v, dtype=float) for v in (a, b, x)))
    MAXIT, EPS, FPMIN = 300, 3e-14, 1e-300
    qab, qap, qam = a + b, a + 1, a - 1
    floor = lambda v: np.where(np.abs(v) < FPMIN, FPMIN, v)
    c = np.ones_like(x)
    d = 1 / floor(1 - (qab * x) / qap)
    h_ = d.copy()
    live = np.ones(x.shape, dtype=bool)
    with np.errstate(all='ignore'):
        for m in range(1, MAXIT + 1):
            m2 = 2 * m
            aa = (m * (b - m) * x) / ((qam + m2) * (a + m2))
            d1 = 1 / floor(1 + aa * d)
            c1 = floor(1 + aa / c)
            h1 = h_ * (d1 * c1)
            aa = (-(a + m) * (qab + m) * x) / ((a + m2) * (qap + m2))
            d1 = 1 / floor(1 + aa * d1)
            c1 = floor(1 + aa / c1)
            de = d1 * c1
            h1 = h1 * de
            c, d, h_ = np.where(live, c1, c), np.where(live, d1, d), np.where(live, h1, h_)
            live &= ~(np.abs(de - 1) < EPS)
            if not live.any():
                break
    return h_


def ibeta(x, a, b):
    x, a, b = np.broadcast_arrays(*(np.atleast_1d(np.asarray(v, dtype=float)) for v in (x, a, b)))
    out = np.where(x <= 0, 0.0, 1.0)
    mid = (x > 0) & (x < 1)
    if mid.any():
        xm, am, bm = x[mid], a[mid], b[mid]
        bt = np.exp(lgamma(am + bm) - lgamma(am) - lgamma(bm) + am * np.log(xm) + bm * np.log(1 - xm))
        front = xm < (am + 1) / (am + bm + 2)
        r = np.empty_like(xm)
        if front.any():
            r[front] = (bt[front] * betacf(am[front], bm[front], xm[front])) / am[front]
        if (~front).any():
            r[~front] = 1 - (bt[~front] * betacf(bm[~front], am[~front], 1 - xm[~front])) / bm[~front]
        out[mid] = r
    return out


def t_sf(t, df):
    t, df = np.atleast_1d(np.asarray(t, dtype=float)), np.atleast_1d(np.asarray(df, dtype=float))
    p = 0.5 * ibeta(df / (df + t * t), df / 2, 0.5)
    return np.where(t >= 0, p, 1 - p)


def f_sf(f, d1, d2):
    f = np.atleast_1d(np.asarray(f, dtype=float))
    with np.errstate(all='ignore'):
        p = ibeta(d2 / (d2 + d1 * f), d2 / 2, d1 / 2)
    return np.where(f <= 0, 1.0, p)


def var_of(n, s, ss):
    return max(0.0, (ss - (s * s) / n) / (n - 1)) if n > 1 else 0.0


def anova(n, s, ss):
    keep_ = n > 0
    n, s, ss = n[keep_].tolist(), s[keep_].tolist(), ss[keep_].tolist()
    k, N_ = len(n), sum(n)
    if k < 2 or N_ - k < 1:
        return None
    grand = sum(s) / N_
    ssb = sum(ni * (si / ni - grand) ** 2 for ni, si in zip(n, s))
    ssw = sum(ssi - ni * (si / ni) ** 2 for ni, si, ssi in zip(n, s, ss))
    if ssw <= 0:
        return None
    d1, d2 = k - 1, N_ - k
    F = ssb / d1 / (ssw / d2)
    return {'F': F, 'd1': d1, 'd2': d2, 'p': float(f_sf(F, d1, d2)[0])}


def welch(na, sa, ssa, nb, sb, ssb):
    """a가 b보다 큰지 단측 Welch t (요약통계 배열)"""
    na, sa, ssa, nb, sb, ssb = (np.atleast_1d(np.asarray(v, dtype=float)) for v in (na, sa, ssa, nb, sb, ssb))
    t, df, p = np.zeros_like(na), np.ones_like(na), np.ones_like(na)
    ok = (na >= 2) & (nb >= 2)
    with np.errstate(all='ignore'):
        va = np.maximum(0, (ssa - (sa * sa) / na) / (na - 1)) / na
        vb = np.maximum(0, (ssb - (sb * sb) / nb) / (nb - 1)) / nb
        se = np.sqrt(va + vb)
        ok &= se != 0
        t[ok] = (sa[ok] / na[ok] - sb[ok] / nb[ok]) / se[ok]
        df[ok] = (va[ok] + vb[ok]) ** 2 / ((va[ok] * va[ok]) / (na[ok] - 1) + (vb[ok] * vb[ok]) / (nb[ok] - 1))
    if ok.any():
        p[ok] = t_sf(t[ok], df[ok])
    return t, df, p


def quantile(v, p):
    if not len(v):
        return math.nan
    h_ = (len(v) - 1) * p
    lo = math.floor(h_)
    return float(v[lo] + (h_ - lo) * (v[min(lo + 1, len(v) - 1)] - v[lo]))

In [ ]:
def combo_key(s, items):
    return f'{s}|' + '|'.join(f'{q}:{u}' for q, u in items)


def binom_fix(n, x, p0):
    """bad(0/1) z의 꼬리 보정 배수. bad 비율이 낮고 웨이퍼가 적으면 bad 장수 분포가 오른쪽으로 길어서
    정규 근사 z가 실제보다 커진다. 정확한 이항 꼬리확률 P(bad ≥ x)로 구한 z ÷ 정규 근사 z를 곱해 줄인다(1보다 크게 하지는 않음)"""
    n, x = np.asarray(n, dtype=float), np.rint(np.asarray(x, dtype=float))
    z_norm = (x - n * p0) / np.sqrt(n * p0 * (1 - p0))
    fix = np.ones_like(n)
    up = z_norm > 0.5
    if up.any() and 0 < p0 < 1:
        nu, xu = n[up], x[up]
        ge = lambda k: np.where(k <= 0, 1.0, ibeta(np.full_like(nu, p0), np.maximum(k, 1), np.maximum(nu - k + 1, 1e-9)))
        fix[up] = np.clip(np.minimum(norm_isf(np.maximum(ge(xu), 1e-300)), 38.0) / z_norm[up], 0.0, 1.0)
    return fix


def shrink(n, N):
    """√(1 − 웨이퍼 수 ÷ 전체 웨이퍼 수). 전체 평균에는 조합의 웨이퍼도 들어 있어서, 조합 평균이 전체 평균에서
    우연히 벗어나는 폭은 σ/√n보다 이만큼 좁다. z를 이 값으로 나누면 조합을 나머지 웨이퍼와 비교한 z가 된다"""
    return np.sqrt(np.clip(1 - np.asarray(n, dtype=float) / N, 1 / N, None))


def analyze(data, min_n=20, max_depth=3, full_depth=2):
    """조합 생성과 조합별 지표 (웹 js/analysis.js analyze와 같은 조합 · 관문 로직).
    웨이퍼(root_lot_id + wafer_id) 한 장을 하나의 표본으로 보고, 신호(Value, bad)마다 조합을 나머지 웨이퍼와 비교한
    z = (조합 평균 − 전체 평균) ÷ (표준편차 ÷ √웨이퍼 수) ÷ √(1 − 웨이퍼 수 ÷ 전체 웨이퍼 수)를 구해 결합 z = (z_Value + z_bad) ÷ √(2 + 2r)로 판정한다.
    bad z는 정확한 이항 꼬리확률로 맞추고, r은 두 z가 우연히 같이 움직이는 정도(Order 1개 조합에서 잰 상관, 0~0.95)라
    같은 정보를 두 번 세지 않게 한다. good_bad가 없으면 z_Value만 쓴다"""
    N, steps, assign = data['N'], data['steps'], data['assign']
    sig = {'y': data['Y']} | ({'b': data['B']} if data.get('B') is not None else {})
    base = {k_: (float(v.mean()), float(v.std(ddof=1))) for k_, v in sig.items()}
    Y = sig['y']
    Y2 = Y * Y
    analysis_y = data.get('analysis_Y')  # 표시 평균만 계산. sig/base/부모 z에는 넣지 않는다.
    mu, sd = base['y']
    bad01 = np.nan_to_num(data['bad']) if 'b' in sig else None
    p0 = float(bad01.mean()) if 'b' in sig else 0.0

    def cell_keys(s, qs):
        A = assign[s][list(qs)]
        dims = [len(steps[s]['seqs'][q]['units']) for q in qs]
        ok = np.all(A != MISSING, axis=0)
        key = np.zeros(N, dtype=np.int64)
        for i, d in enumerate(dims):
            key = key * d + A[i]
        return dims, ok, key

    def cells(s, qs):
        """오더 묶음 qs에서 웨이퍼가 실제로 지난 칸(유닛 조합)만: 칸 번호(오름차순) · 칸마다 웨이퍼 수 · 웨이퍼 → 칸 순번.
        가능한 칸 수(유닛 수의 곱)가 아무리 커도 웨이퍼 수만큼만 계산한다"""
        dims, ok, key = cell_keys(s, qs)
        values = key[ok]
        space = math.prod(dims)
        # 작은 정수 공간은 정렬 대신 직접 집계. 원본과 같은 오름차순 u / inv / cnt를 반환한다.
        if space <= 1_000_000 and space <= max(4096, 4 * len(values)):
            dense_counts = np.bincount(values, minlength=space)
            u = np.flatnonzero(dense_counts)
            cnt = dense_counts[u]
            lookup = np.empty(space, dtype=np.intp)
            lookup[u] = np.arange(len(u), dtype=np.intp)
            inv = lookup[values]
        else:
            u, inv, cnt = np.unique(values, return_inverse=True, return_counts=True)
            inv = np.asarray(inv, dtype=np.intp).reshape(-1)
        return dims, ok, u, inv, cnt

    def stats(ok, inv, m, idx):
        """칸마다 신호별 합을 구해 idx 칸만 돌려준다"""
        w = lambda v: np.bincount(inv, weights=v[ok], minlength=m)[idx]
        out = {'ss': w(Y2)} | {f'sum_{k_}': w(v) for k_, v in sig.items()}
        if analysis_y is not None:
            out['sum_analysis'] = w(analysis_y)
        if 'b' in sig:
            out['x_bad'] = w(bad01)                   # bad 장수 (0/1): 이항분포 꼬리 보정용
        return out

    def sig_z(k_, nn, col):
        """조합들의 신호 k_ z: (조합 평균 − 전체 평균) ÷ (표준편차 ÷ √웨이퍼 수) ÷ √(1 − 웨이퍼 수 ÷ 전체 웨이퍼 수)"""
        m_, s_ = base[k_]
        return (col(f'sum_{k_}') / nn - m_) / (s_ / np.sqrt(nn)) / shrink(nn, N)

    def decode(qs, dims, key):
        units = []
        for d in reversed(dims):
            units.append(key % d)
            key //= d
        return tuple((q, int(u)) for q, u in zip(qs, reversed(units)))

    def parent_keys(dims, keys, i):
        """칸 번호에서 i번째 오더 자리를 뺀 부모 칸 번호"""
        low = math.prod(dims[i + 1:])
        return (keys // (low * dims[i])) * low + keys % low

    seq_info = []
    for s, st in enumerate(steps):
        rows = []
        for q, sq in enumerate(st['seqs']):
            a = assign[s][q]
            ok, nu = a != MISSING, len(sq['units'])
            n_u = np.bincount(a[ok], minlength=nu)
            s_u, ss_u = np.bincount(a[ok], weights=Y[ok], minlength=nu), np.bincount(a[ok], weights=Y2[ok], minlength=nu)
            units = [{'name': name, 'n': int(n_u[u]), 'mean': s_u[u] / n_u[u] if n_u[u] else math.nan,
                      'sd': math.sqrt(var_of(int(n_u[u]), s_u[u], ss_u[u]))} for u, name in enumerate(sq['units'])]
            testable = nu >= 2
            rows.append({'name': sq['name'], 'units': units, 'testable': testable, 'anova': anova(n_u, s_u, ss_u) if testable else None})
        seq_info.append(rows)

    combos, by_key, r = [], {}, 0.0
    flagged = {}                        # (스텝, 오더 묶음) → z 1.96을 넘은 칸 번호들 (다음 깊이의 관문)
    has_big = set()                     # 웨이퍼 MIN_N장 이상인 칸이 있는 (스텝, 오더 묶음). 부모에 없으면 자식에도 없다
    multi = [[q for q, sq in enumerate(st['seqs']) if len(sq['units']) >= 2] for st in steps]
    search = [0] * (max_depth + 1)
    progress_time = time.perf_counter()
    for k in range(1, max_depth + 1):
        gated = k > full_depth
        cand = []
        for s in range(len(steps)):
            if len(multi[s]) < k:
                continue
            for qs in combinations(multi[s], k):
                if SHOW_PROGRESS and time.perf_counter() - progress_time >= 5:
                    print(f'    깊이 {k}/{max_depth} · STEP {s+1}/{len(steps)} · 후보 {len(combos):,}개', flush=True)
                    progress_time = time.perf_counter()
                subs = [(s, qs[:i] + qs[i + 1:]) for i in range(k)]
                if k > 1 and not all(p in has_big for p in subs):
                    continue                    # 오더 하나를 뺀 묶음에 큰 칸이 없으면 이 묶음에도 없다
                dims, ok, u, inv, cnt = cells(s, qs)
                big = np.flatnonzero(cnt >= min_n)
                search[k] += len(big)           # 기준 z의 보정에는 계산을 건너뛴 칸도 센다
                if len(big):
                    has_big.add((s, qs))
                if not len(big) or (gated and not any(p in flagged for p in subs)):
                    continue
                if gated:                       # 오더 하나를 뺀 부모 조합 중 하나라도 z 1.96을 넘은 칸만 계산
                    keep = np.zeros(len(big), dtype=bool)
                    for i, p in enumerate(subs):
                        if p in flagged:
                            keep |= np.isin(parent_keys(dims, u[big], i), flagged[p])
                    big = big[keep]
                    if not len(big):
                        continue
                cand.append((s, qs, dims, u[big], cnt[big], stats(ok, inv, len(u), big)))
        if not cand:
            continue
        col = lambda f: np.concatenate([c[5][f] for c in cand]).astype(float)
        nn = np.concatenate([c[4] for c in cand]).astype(float)
        sm, ss = col('sum_y'), col('ss')
        analysis_means = col('sum_analysis') / nn if analysis_y is not None else None
        zy = sig_z('y', nn, col)
        zb = sig_z('b', nn, col) * binom_fix(nn, col('x_bad'), p0) if 'b' in sig else None
        if zb is not None and k == 1 and len(zy) > 2:   # 두 z의 상관: 오더 1개 조합(대부분 우연)에서 잰다
            r = float(np.clip(np.corrcoef(zy, zb)[0, 1], 0.0, 0.95))
        z = zy if zb is None else (zy + zb) / math.sqrt(2 + 2 * r)
        bad_rate = col('x_bad') / nn if 'b' in sig else None   # 원래(0/1) bad 비율 — 랭킹의 초과 bad 웨이퍼 계산용
        j = 0
        for s, qs, dims, keys, _, _ in cand:
            hot = []
            for key in keys.tolist():
                items = decode(qs, dims, key)
                c = {'id': len(combos), 'key': combo_key(s, items), 'step': s, 'k': k, 'items': items,
                     'n': int(nn[j]), 'sum': float(sm[j]), 'ss': float(ss[j]), 'mean': float(sm[j] / nn[j]),
                     'z': float(z[j]), 'z_y': float(zy[j]), 'z_b': float(zb[j]) if zb is not None else math.nan,
                     'bad_rate': float(bad_rate[j]) if bad_rate is not None else math.nan}
                if analysis_means is not None:
                    c['analysis_mean'] = float(analysis_means[j])
                combos.append(c)
                by_key[c['key']] = c
                if c['z'] > Z95:
                    hot.append(key)
                j += 1
            if hot:
                flagged[(s, qs)] = np.array(hot, dtype=np.int64)
    depth_count = [0] * (max_depth + 1)
    for c in combos:
        depth_count[c['k']] += 1
    return {'mu': mu, 'sd': sd, 'N': N, 'max_depth': max_depth, 'signals': list(sig), 'r': r, 'p0': p0 if 'b' in sig else None,
            'seq_info': seq_info, 'combos': combos, 'by_key': by_key, 'depth_count': depth_count, 'search_space': search}



In [ ]:
POINT_COLUMNS = ['part_id', 'step', 'depth', 'path', 'n', 'sample_mean']

def mean_envelope(points, mu_hat, sigma_hat, *, sigma_hat_raw):
    # 선택한 분석 단위의 웨이퍼 표준편차를 모든 n에 공통으로 사용한다.
    # sigma_hat_raw는 원단위 참고값이며 ON의 경계 계산에는 사용하지 않는다.
    summary_columns = ['n', 'm_n', 'mean_of_means', 'sigma_hat', 'sigma_hat_raw', 'sigma_hat_n',
                       'mu_hat', 'sqrt_2log_m', 'extreme_coefficient', 'coefficient_method', 'half_width', 'lower', 'upper', 'observed_min', 'observed_max']
    if points.empty:
        summary = pd.DataFrame(columns=summary_columns)
        result = points.copy()
        for col in summary_columns:
            if col not in ('n', 'mean_of_means', 'observed_min', 'observed_max'):
                result[col] = pd.Series(dtype=float)
        result['outside'] = pd.Series(dtype='boolean')
        result['direction'] = pd.Series(dtype=str)
        return result, summary
    summary = (points.groupby('n', sort=True)['sample_mean']
               .agg(m_n='size', mean_of_means='mean', observed_min='min', observed_max='max')
               .reset_index())
    summary['sigma_hat_raw'] = sigma_hat_raw
    summary['sigma_hat'] = sigma_hat
    summary['sigma_hat_n'] = sigma_hat / np.sqrt(summary['n'])
    summary['mu_hat'] = mu_hat
    summary['sqrt_2log_m'] = np.sqrt(2 * np.log(summary['m_n']))
    # 같은 m의 정확한 정규 기대 최대 절대편차는 캐시하여 재사용한다.
    # 임의의 m 절단점 없이 모든 m에 같은 이론을 적용한다. m=1에서도 양의 폭이다.
    summary['extreme_coefficient'] = (
        summary['m_n'].map(expected_normal_absolute_max)
        if EXTREME_COEFFICIENT == 'absolute_max' else summary['sqrt_2log_m'])
    summary['coefficient_method'] = EXTREME_COEFFICIENT
    half_width = summary['sigma_hat_n'] * summary['extreme_coefficient']
    summary['half_width'] = half_width
    summary['lower'] = mu_hat - half_width
    summary['upper'] = mu_hat + half_width
    result = points.merge(summary.drop(columns=['mean_of_means', 'observed_min', 'observed_max']), on='n', how='left', validate='many_to_one')
    # 경계와 소수점 연산 오차 수준으로 같은 값은 경계 위로 취급한다. 경계 수식 자체는 바꾸지 않는다.
    scale = result[['sample_mean', 'lower', 'upper']].abs().max(axis=1)
    rounding_tolerance = 32 * np.finfo(float).eps * scale
    above = result['sample_mean'] > result['upper'] + rounding_tolerance
    below = result['sample_mean'] < result['lower'] - rounding_tolerance
    result['outside'] = (above | below).astype('boolean')
    result['direction'] = np.select([above, below], ['위쪽', '아래쪽'], default='경계 안')
    return result, summary

def step_sort_key(value):
    return (0, float(value), value) if value.replace('.', '', 1).isdigit() else (1, 0.0, value)


def prepare_reference_part(part_raw, part_id):
    # 원본처럼 first()는 결측을 건너뛴 첫 값을 사용한다.
    wafer_all = (part_raw.group_by(WAFER_KEY).agg(
        pl.col('_y').drop_nulls().first().alias('raw_y'),
        pl.col('_y').drop_nulls().n_unique().alias('y_n'),
        pl.col('_bad').drop_nulls().first().alias('bad'),
        pl.col('_gb').drop_nulls().first().alias('gb'),
    ).sort(WAFER_KEY))
    if wafer_all['raw_y'].null_count():
        raise ValueError(f'part {part_id}: raw y가 없는 웨이퍼 {wafer_all["raw_y"].null_count()}장. 입력을 확인하세요.')
    has_bad = wafer_all['bad'].null_count() < wafer_all.height
    wafer_used = wafer_all.filter(pl.col('bad').is_not_null()) if has_bad else wafer_all
    wafer_used = wafer_used.with_row_index('_wi')
    n = wafer_used.height
    raw_y = wafer_used['raw_y'].to_numpy()
    if n < 2 * MIN_N:
        raise ValueError(f'원본 최소 비교 조건 미달: 웨이퍼 {n}장 < {2 * MIN_N}장')
    if not float(np.std(raw_y)) > 0:
        raise ValueError('원본 비교 조건 미달: raw y 표준편차가 0입니다.')
    history = part_raw.filter(pl.col('_step').is_not_null() & pl.col('_order').is_finite() & pl.col('eqp_id').is_not_null())
    history = history.join(wafer_used.select(WAFER_KEY + ['_wi']), on=WAFER_KEY, how='inner')
    before = history.height
    history = (history.sort(['_time', '_row'], nulls_last=(REWORK != 'last'))
               .unique(subset=['_wi', '_step', '_order'], keep=REWORK, maintain_order=True))
    duplicates = before - history.height
    if history.is_empty():
        raise ValueError('유효한 공정 이력이 없습니다.')
    # 유닛 이름·정렬 순서는 원본 eqp_id-chamber_id 방식과 같다.
    history = history.with_columns(pl.when(pl.col('chamber_id') != '')
        .then(pl.concat_str(['eqp_id', 'chamber_id'], separator='-'))
        .otherwise(pl.col('eqp_id')).alias('_unit'))
    step_groups = history.partition_by('_step', as_dict=True)
    steps, assign = [], []
    for step_key in sorted(step_groups, key=lambda key: step_sort_key(str(key[0]))):
        name, step_history = str(step_key[0]), step_groups[step_key]
        orders = sorted(step_history['_order'].unique().to_list())
        a = np.full((len(orders), n), MISSING, dtype=np.uint16)
        seqs = []
        for qi, order in enumerate(orders):
            h = step_history.filter(pl.col('_order') == order)
            units = sorted(h['_unit'].unique().to_list())
            if len(units) >= MISSING:
                raise ValueError('한 Order의 유닛 수가 너무 많습니다.')
            codes = h['_unit'].replace_strict(dict(zip(units, range(len(units)))), return_dtype=pl.UInt16).to_numpy()
            a[qi, h['_wi'].to_numpy()] = codes
            label = str(int(order)) if float(order).is_integer() else str(order)
            seqs.append({'name': f'O{label}', 'ord': float(order), 'units': units})
        steps.append({'index': len(steps), 'name': name, 'seqs': seqs})
        assign.append(a)
    bad = wafer_used['bad'].to_numpy()
    b = bad.copy() if has_bad else None
    if b is not None and b.std() == 0:
        b = None
    data = {'N': n, 'Y': raw_y.copy() if HIGHER_IS_WORSE else -raw_y,
            'B': b, 'bad': bad, 'steps': steps, 'assign': assign}
    wf = pd.DataFrame(wafer_used.select(WAFER_KEY + ['raw_y']).to_dict(as_series=False)).set_index(WAFER_KEY)
    quality = {'wafer_total': wafer_all.height, 'excluded_non_gb': wafer_all.height - n,
               'multiple_y': int((wafer_all['y_n'] > 1).sum()), 'duplicate_history_rows': duplicates}
    return wf, data, quality


"""Robust lambda fitting for Yeo–Johnson applied directly to raw observations.

Independent NumPy/SciPy implementation based on Raymaekers & Rousseeuw,
Transforming variables to central normality, doi:10.1007/s10994-021-05960-5,
Sections 2.1–2.4. Unlike their recommended prestandardized setup, the input
here is NOT centered/scaled before Yeo–Johnson. Rectification is used only
in the initial fit. The final transformation is ordinary Yeo–Johnson.

The authors' R cellWise implementation is GPL >= 2; no R/C++ source is
vendored or translated here. Numerical choices are documented below.
"""
import numpy as np


def _yj_huber_location_scale(values):
    """One-step Huber M location and scale, initialized by median / normal MAD.

    k=1.5 for both estimates. The scale consistency constant is evaluated
    analytically as E[min(Z**2, k**2)] for Z standard normal. These internal
    robust estimates fit lambda only; they are not the chart's mean / SD.
    """
    from scipy.special import ndtr, ndtri

    z = np.asarray(values, dtype=float)
    if z.ndim != 1 or not np.isfinite(z).all():
        raise ValueError("Yeo-Johnson fit encountered non-finite transformed values")
    location = float(np.median(z))
    scale0 = float(np.median(np.abs(z - location))) / ndtri(0.75)
    if not np.isfinite(scale0) or scale0 <= 0:
        raise ValueError("MAD is zero: robust lambda fitting is undefined for this part")
    k = 1.5
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        residual = (z - location) / scale0
        weights = np.minimum(1.0, k / np.maximum(np.abs(residual), np.finfo(float).tiny))
        location += scale0 * float(np.mean(np.clip(residual, -k, k))) / float(weights.mean())
    if not np.isfinite(location):
        raise ValueError("Huber location is numerically non-finite")
    residual_abs = np.abs(z - location)
    scale0 = float(np.median(residual_abs)) / ndtri(0.75)
    if not np.isfinite(scale0) or scale0 <= 0:
        raise ValueError("MAD is zero after Huber location fitting")
    # Integral z^2 phi(z) on [-k,k], plus the two winsorized tails.
    tail = float(ndtr(-k))
    density = float(np.exp(-0.5 * k * k) / np.sqrt(2 * np.pi))
    normal_factor = 1 - 2 * (k * density + tail) + 2 * k * k * tail
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        clipped_residual = np.minimum(residual_abs / scale0, k)
        scale = scale0 * np.sqrt(float(np.mean(clipped_residual**2)) / normal_factor)
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError("Huber scale is zero or numerically non-finite")
    return location, float(scale)


def _yj_rectified_for_fit(x_sorted, lmbda, quartiles):
    """Paper equations (9),(10), with quartile knots restricted to their branch.

    Cl=min(Q1,0), Cu=max(Q3,0) allows uncentered raw input while retaining
    the required signs of the rectification knots. No data is removed.
    """
    from scipy.stats import yeojohnson

    low, high = quartiles
    if lmbda == 1:
        return x_sorted.copy()
    knot = high if lmbda < 1 else low
    linear = x_sorted > knot if lmbda < 1 else x_sorted < knot
    result = np.empty_like(x_sorted)
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        result[~linear] = yeojohnson(x_sorted[~linear], lmbda=lmbda)
        knot_y = float(yeojohnson(np.array([knot]), lmbda=lmbda)[0])
        derivative = np.exp((lmbda - 1) * np.sign(knot) * np.log1p(abs(knot)))
        result[linear] = knot_y + derivative * (x_sorted[linear] - knot)
    return result


def _yj_minimize_bounded(objective, bounds, *, stage):
    """Coarse grid plus local refinements; fail visibly if optimum hits bounds.

    Bounds are numerical search limits, not an assertion about allowed YJ
    lambda. Callers can widen them or handle the exception explicitly.
    """
    from scipy.optimize import minimize_scalar

    lower, upper = map(float, bounds)
    if not np.isfinite([lower, upper]).all() or lower >= upper:
        raise ValueError("Yeo-Johnson lambda bounds must be finite and increasing")
    grid = np.unique(np.r_[np.linspace(lower, upper, 21),
                            [v for v in (0.0, 1.0, 2.0) if lower <= v <= upper]])
    scores = np.array([objective(value) for value in grid])
    finite = np.isfinite(scores)
    if not finite.any():
        raise ValueError(f"{stage}: no numerically valid lambda in {bounds}")
    candidates = [(float(scores[i]), float(grid[i])) for i in np.flatnonzero(finite)]
    # Refine the three best grid neighborhoods to reduce dependence on a
    # single initial bracket in the bounded, potentially nonconvex QQ loss.
    for index in np.argsort(scores)[:3]:
        if not np.isfinite(scores[index]):
            continue
        left, right = grid[max(0, index - 1)], grid[min(len(grid) - 1, index + 1)]
        if left == right:
            continue
        # Invalid lambda candidates deliberately return infinity. Brent's
        # interpolation may perform intermediate inf arithmetic; candidate
        # validation below still rejects non-finite results explicitly.
        with np.errstate(invalid="ignore", over="ignore"):
            fit = minimize_scalar(objective, bounds=(left, right), method="bounded",
                                  options={"xatol": 1e-5, "maxiter": 100})
        if fit.success and np.isfinite(fit.fun):
            candidates.append((float(fit.fun), float(fit.x)))
    score, estimate = min(candidates)
    margin = max(1e-4, (upper - lower) * 1e-4)
    if estimate <= lower + margin or estimate >= upper - margin:
        raise ValueError(f"{stage}: lambda optimum reached search bounds {bounds}; widen bounds")
    return estimate, score


def _yj_minimize_expanding(objective, bounds, *, stage, max_expansions=12):
    """Expand an initial numerical bracket when the best fit reaches an edge.

    Raw rates can need |lambda| far greater than 4 or 6. Expansion changes
    the numerical search interval only, never the input units or location.
    """
    lower, upper = map(float, bounds)
    for attempt in range(max_expansions + 1):
        try:
            estimate, score = _yj_minimize_bounded(objective, (lower, upper), stage=stage)
            return estimate, score, (lower, upper)
        except ValueError as exc:
            if "reached search bounds" not in str(exc) or attempt == max_expansions:
                raise
            left_score, right_score = objective(lower), objective(upper)
            width = upper - lower
            if left_score <= right_score:
                lower -= 2 * width
            else:
                upper += 2 * width
    raise ValueError(f"{stage}: search interval expansion failed")


def fit_rewml_yj(raw_y, *, bounds=(-4.0, 6.0)):
    """Return (lambda, diagnostics) using a RewML-based raw-input adaptation.

    Uses all observations (no sampling), one initial sort, a bounded Tukey
    QQ loss (c=.5), followed by exactly two hard-reweighted ordinary-YJ
    likelihood fits. The .995 normal cutoff selects observations used to
    estimate lambda; it does NOT define the extreme-value chart envelope.

    Only lambda fitting uses robust location/scale and zero/one weights.
    Apply the fitted ordinary YJ to EVERY original wafer afterward, and
    compute chart moments from ALL transformed wafers. Too-small samples,
    zero MAD (often due to many tied zeros), overflow, and unresolved boundary
    optima after adaptive expansion raise ValueError; the caller must report any fallback explicitly.
    """
    from scipy.special import ndtri
    from scipy.stats import yeojohnson

    raw = np.asarray(raw_y, dtype=float)
    if raw.ndim != 1 or len(raw) < 8 or not np.isfinite(raw).all():
        raise ValueError("RewML needs a finite one-dimensional array with at least 8 values")
    x = np.sort(raw)
    _yj_huber_location_scale(x)  # Explicitly reject zero-MAD / degenerate parts.
    quartiles = np.quantile(x, [0.25, 0.75])
    knots = (min(float(quartiles[0]), 0.0), max(float(quartiles[1]), 0.0))
    theoretical = ndtri((np.arange(1, len(x) + 1) - 1/3) / (len(x) + 1/3))
    objective_evaluations = 0

    def qq_objective(lmbda):
        nonlocal objective_evaluations
        objective_evaluations += 1
        try:
            transformed = _yj_rectified_for_fit(x, lmbda, knots)
            location, scale = _yj_huber_location_scale(transformed)
            with np.errstate(over="ignore", invalid="ignore"):
                residual = (transformed - location) / scale - theoretical
                square = np.minimum(np.abs(residual) / 0.5, 1.0)**2
                loss = float(np.mean(1 - (1 - square)**3))
            return loss if np.isfinite(loss) else np.inf
        except (ValueError, FloatingPointError, OverflowError):
            return np.inf

    lmbda, initial_loss, used_bounds = _yj_minimize_expanding(qq_objective, bounds, stage="RewML initial fit")
    initial_lmbda = lmbda
    # First hard weights use the rectified initialization. Further weights
    # use ordinary YJ after each likelihood update. This raw-input
    # adaptation uses rectified initial weights, as in cellWise transfo;
    # the weighted likelihood itself always uses ordinary YJ.
    transformed = _yj_rectified_for_fit(x, lmbda, knots)
    cutoff = float(ndtri(0.995))
    retained = []
    lambda_steps = []
    for step in range(2):
        location, scale = _yj_huber_location_scale(transformed)
        with np.errstate(over="ignore", invalid="ignore"):
            keep = np.abs((transformed - location) / scale) <= cutoff
        fit_x = x[keep]
        if len(fit_x) < 3 or fit_x[0] == fit_x[-1]:
            raise ValueError("RewML weights retained too few distinct observations")
        retained.append(int(keep.sum()))
        log_jacobian_mean = float(np.mean(np.sign(fit_x) * np.log1p(np.abs(fit_x))))

        def weighted_nll(candidate):
            nonlocal objective_evaluations
            objective_evaluations += 1
            with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
                values = yeojohnson(fit_x, lmbda=candidate)
                # Scale before variance to avoid overflow in squared values.
                factor = float(np.max(np.abs(values)))
                if not np.isfinite(factor) or factor == 0:
                    return np.inf
                variance_scaled = float(np.var(values / factor))
                if not np.isfinite(variance_scaled) or variance_scaled <= 0:
                    return np.inf
                log_variance = np.log(variance_scaled) + 2 * np.log(factor)
                value = 0.5 * log_variance - (candidate - 1) * log_jacobian_mean
            return float(value) if np.isfinite(value) else np.inf

        lmbda, _, used_bounds = _yj_minimize_expanding(weighted_nll, used_bounds, stage=f"RewML weighted step {step + 1}")
        lambda_steps.append(lmbda)
        with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
            transformed = yeojohnson(x, lmbda=lmbda)
    location, scale = _yj_huber_location_scale(transformed)
    with np.errstate(over="ignore", invalid="ignore"):
        final_keep = np.abs((transformed - location) / scale) <= cutoff
    return float(lmbda), {
        "method": "rewml_raw",
        "fit_n": int(len(x)),
        "initial_lambda": float(initial_lmbda),
        "lambda_initial": float(initial_lmbda),
        "initial_qq_loss": float(initial_loss),
        "lambda_steps": tuple(float(value) for value in lambda_steps),
        "kept_per_step": tuple(retained),
        "n_kept_last_fit": retained[-1],
        "n_used_last": retained[-1],
        "n_kept_final": int(final_keep.sum()),
        "initial_search_bounds": tuple(map(float, bounds)),
        "bounds_used": tuple(map(float, used_bounds)),
        "objective_evaluations": objective_evaluations,
        "prestandardized": False,
    }


def transform_part_y(raw_y, enabled, *, method=None, return_info=False):
    """λ 추정만 선택한다. 출력은 raw 입력에 대한 일반 YJ이며 표준화하지 않는다."""
    if not isinstance(enabled, (bool, np.bool_)):
        raise ValueError('use_yeo_johnson은 True 또는 False여야 합니다.')
    raw_y = np.asarray(raw_y, dtype=float)
    if raw_y.ndim != 1 or raw_y.size < 2 or not np.isfinite(raw_y).all():
        raise ValueError('변환 입력은 유한한 값이 2개 이상인 1차원 웨이퍼 배열이어야 합니다.')
    if np.all(raw_y == raw_y[0]):
        raise ValueError('raw y가 상수이므로 표준편차와 변환을 추정할 수 없습니다.')
    requested = YJ_LAMBDA_METHOD if method is None else method
    if requested not in ('robust', 'mle'):
        raise ValueError("lambda_method는 'robust' 또는 'mle'여야 합니다.")
    info = {'requested': requested if enabled else 'raw', 'used': 'raw',
            'status': 'off', 'note': '', 'n_fit': len(raw_y), 'n_used_last': len(raw_y)}
    if not enabled:
        result = (raw_y.copy(), None)
        return (*result, info) if return_info else result
    try:
        from scipy.stats import yeojohnson
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError('Yeo–Johnson ON에는 SciPy가 필요합니다. %pip install scipy 후 다시 실행하세요.') from exc
    info.update(used=requested, status='ok')
    try:
        if requested == 'robust':
            try:
                fitted_lambda, details = fit_rewml_yj(raw_y)
                info.update(details)
                values = yeojohnson(raw_y, lmbda=fitted_lambda)
            except (ValueError, RuntimeError, OverflowError, FloatingPointError) as exc:
                if YJ_ROBUST_FALLBACK == 'error':
                    raise ValueError(f'강건한 λ 추정 불가: {exc}') from exc
                info.update(used='mle', status='fallback', note=str(exc), n_used_last=len(raw_y))
                values, fitted_lambda = yeojohnson(raw_y)
        else:
            values, fitted_lambda = yeojohnson(raw_y)
    except (ValueError, RuntimeError, OverflowError, FloatingPointError) as exc:
        raise ValueError(f'Yeo–Johnson 적합 실패: {exc}') from exc
    if not np.isfinite(fitted_lambda) or not np.isfinite(values).all():
        raise ValueError('Yeo–Johnson 결과에 비유한 값이 있습니다. 입력 범위를 확인하세요.')
    sigma = float(np.std(values, ddof=1))
    if not np.isfinite(sigma) or sigma <= 0:
        raise ValueError('Yeo–Johnson 결과의 표준편차가 0 또는 비유한 값입니다.')
    result = (values, float(fitted_lambda))
    return (*result, info) if return_info else result


def rank_normal_part_y(raw_y):
    """Part별 고유 웨이퍼를 평균순위 정규점수로 변환. 동률/입력 순서 불변."""
    raw_y = np.asarray(raw_y, dtype=float)
    if raw_y.ndim != 1 or raw_y.size < 2 or not np.isfinite(raw_y).all():
        raise ValueError('순위 변환 입력은 유한한 값이 2개 이상인 1차원 배열이어야 합니다.')
    unique, counts = np.unique(raw_y, return_counts=True)
    if len(unique) < 2:
        raise ValueError('raw y가 상수이므로 순위 점수의 표준편차가 0입니다.')
    try:
        from scipy.stats import rankdata
        from scipy.special import ndtri
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError('순위 정규변환에는 SciPy가 필요합니다. %pip install scipy 후 다시 실행하세요.') from exc
    ranks = rankdata(raw_y, method='average')
    values = ndtri((ranks - 0.5) / len(raw_y))
    tied_fraction = float(counts[counts > 1].sum() / len(raw_y))
    max_tie_fraction = float(counts.max() / len(raw_y))
    note = (f'동일 y 최대 집단 {max_tie_fraction:.1%}: 동률에 같은 점수를 부여하므로 정규분포와 다를 수 있습니다.'
            if len(unique) < len(raw_y) else '')
    return values, {'requested': 'rank_normal', 'used': 'rank_normal', 'status': 'ok',
                    'note': note, 'n_fit': len(raw_y), 'n_used_last': len(raw_y),
                    'n_unique': len(unique), 'tied_fraction': tied_fraction,
                    'max_tie_fraction': max_tie_fraction, 'ties': 'average'}


def analysis_value_label(report):
    return {'raw': 'raw y', 'yeo-johnson': 'Yeo–Johnson y',
            'rank_normal': 'Rank-normal score'}[report['transform']]


def analysis_mode_label(report):
    if report['transform'] == 'raw':
        return 'Raw · transform OFF'
    if report['transform'] == 'rank_normal':
        return 'Rank-normal · average ties'
    fit = report['transform_fit']
    label = f"Yeo–Johnson {fit['used']} · λ={report['transform_lambda']:.6g}"
    return label + (' (fallback)' if fit['status'] == 'fallback' else '')


"""Empirical scale calibration from paired normal order statistics.

The final plotting envelope is deliberately NOT computed here.  Callers retain
their baseline columns and use ``fit_info['sigma_effective']`` only when status
is ``'ok'``.  Huber calibration is an empirical effective scale: normal maxima
have skewed errors, so this is not an unbiased estimator of the true sigma.
"""

from functools import lru_cache
import math


@lru_cache(maxsize=8192)
def expected_normal_max_moments(m: int) -> tuple[float, float]:
    """Return E[max Z_j] and Var(max Z_j), Z_j iid standard normal.

    Integrate the exact order-statistic distribution, not a Gumbel expansion.
    SciPy is imported only on demand.  Log-CDF calculations protect large m;
    the variance integral is centered to avoid subtracting two large moments.
    """
    try:
        count = int(m)
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError('m must be a positive integer') from exc
    if isinstance(m, bool) or count < 1 or count != m:
        raise ValueError('m must be a positive integer')
    if count == 1:
        return 0.0, 1.0
    mf = float(count)
    if not math.isfinite(mf):
        raise ValueError('m is too large for floating-point quadrature')

    from scipy.integrate import quad
    from scipy.special import log_ndtr

    def tails(x):
        log_f = mf * float(log_ndtr(x))
        return -math.expm1(log_f), math.exp(log_f)

    def mean_integrand(t):
        upper, _ = tails(t)
        _, below_minus_t = tails(-t)
        return upper - below_minus_t

    # Splitting near the maximum prevents an infinite-interval integrator from
    # missing the transition when m is very large.
    split = math.sqrt(2.0 * math.log(mf))
    options = dict(epsabs=2e-11, epsrel=2e-11, limit=250)
    mean = (quad(mean_integrand, 0.0, split, **options)[0]
            + quad(mean_integrand, split, math.inf, **options)[0])

    def variance_integrand(t):
        upper, _ = tails(mean + t)
        _, lower = tails(mean - t)
        return 2.0 * t * (upper + lower)

    # Equivalent to E[M^2] - E[M]^2, evaluated as E[(M - E[M])^2].
    v_split = max(0.25, 1.0 / (1.0 + mean))
    variance = (quad(variance_integrand, 0.0, v_split, **options)[0]
                + quad(variance_integrand, v_split, math.inf, **options)[0])
    if not (math.isfinite(mean) and mean > 0.0
            and math.isfinite(variance) and variance > 0.0):
        raise ArithmeticError(f'normal maximum moments failed for m={count}')
    return float(mean), float(variance)


"""Exact finite-m expected absolute maximum for a two-sided Gaussian envelope."""

from functools import lru_cache
import math


@lru_cache(maxsize=8192)
def expected_normal_absolute_max(m: int) -> float:
    """Return E[max_{j<=m} |Z_j|] for iid Z_j ~ N(0, 1).

    This is the expected largest absolute deviation, not a confidence level and
    not E[max Z_j].  Integrates the exact finite-m survival function
        1 - (2 * Phi(t) - 1)**m,  t >= 0.
    The m=1 value is sqrt(2/pi), so a singleton's display width is positive.
    SciPy is imported locally, only when a numerical integral is needed.
    """
    try:
        count = int(m)
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError('m must be a positive integer') from exc
    if isinstance(m, bool) or count < 1 or count != m:
        raise ValueError('m must be a positive integer')
    if count == 1:
        return math.sqrt(2.0 / math.pi)
    try:
        mf = float(count)
    except OverflowError as exc:
        raise ValueError('m is too large for floating-point quadrature') from exc
    if not math.isfinite(mf):
        raise ValueError('m is too large for floating-point quadrature')

    from scipy.integrate import quad
    from scipy.special import log_ndtr

    log_m = math.log(mf)
    log_two = math.log(2.0)
    sqrt_two = math.sqrt(2.0)

    def survival(t):
        if t <= 0.0:
            return 1.0
        if t < 1.0:
            # erf is accurate near zero, where 2*Phi(t)-1 would cancel.
            log_abs_cdf = math.log(math.erf(t / sqrt_two))
            return -math.expm1(mf * log_abs_cdf)
        log_abs_tail = log_two + float(log_ndtr(-t))
        if log_abs_tail < -36.0:
            # -log(1-q) = q*(1+O(q)); here relative error < 1.2e-16.
            # Form m*q in log space so a tiny q cannot underflow before being
            # multiplied by a very large m.
            log_intensity = log_m + log_abs_tail
            if log_intensity > math.log(745.0):
                return 1.0
            return -math.expm1(-math.exp(log_intensity))
        tail = math.exp(log_abs_tail)
        return -math.expm1(mf * math.log1p(-tail))

    split = math.sqrt(2.0 * (log_m + log_two))
    options = dict(epsabs=2e-11, epsrel=2e-11, limit=250)
    value = (quad(survival, 0.0, split, **options)[0]
             + quad(survival, split, math.inf, **options)[0])
    if not math.isfinite(value) or value <= 0.0:
        raise ArithmeticError(f'absolute-normal maximum integration failed for m={count}')
    return float(value)


def empirical_extreme_fit(summary, mu, sigma0, *, enabled=True, min_groups=5,
                          huber_delta=1.5, method='huber'):
    """Fit one nonnegative effective scale multiplier for one part.

    Required summary columns: n, m_n, observed_min, observed_max; one row per n.
    Both extrema receive one loss term per n, scaled by the fixed theoretical
    SD of a standard-normal maximum.  The Huber objective is
        sum Huber((d_plus/minus - kappa * a_m) / max_sd).
    ``method='wls'`` uses squared loss with those same fixed scales.

    d_plus and d_minus remain signed: a whole group can lie on one side of mu.
    m_n < 2 carries no information about the expected maximum scale and is
    excluded.  Unavailable fits return NaN, never a silent kappa=1 fallback.
    No standard errors are claimed: paired minima/maxima are correlated, and
    Huber's nonlinear score need not be mean-consistent for skewed extrema.
    """
    import numpy as np
    import pandas as pd

    required = {'n', 'm_n', 'observed_min', 'observed_max'}
    missing = required.difference(summary.columns)
    if missing:
        raise ValueError(f'missing summary columns: {sorted(missing)}')
    if method not in {'huber', 'wls'}:
        raise ValueError("method must be 'huber' or 'wls'")
    if (isinstance(min_groups, bool) or int(min_groups) != min_groups
            or min_groups < 1):
        raise ValueError('min_groups must be a positive integer')
    if not math.isfinite(float(huber_delta)) or huber_delta <= 0:
        raise ValueError('huber_delta must be finite and positive')
    delta = float(huber_delta)
    result = summary.copy()
    float_columns = (
        'a_m', 'max_sd', 'd_plus', 'd_minus',
        'fit_expected_upper', 'fit_expected_lower',
        'fit_resid_plus', 'fit_resid_minus',
        'huber_weight_plus', 'huber_weight_minus',
    )
    for column in float_columns:
        result[column] = np.nan
    result['fit_eligible'] = False
    result['fit_used'] = False
    info = {
        'status': 'unavailable', 'message': '', 'method': method,
        'interpretation': 'empirical_effective_scale_not_unbiased_sigma',
        'kappa': math.nan, 'kappa_wls': math.nan,
        'kappa_wls_unconstrained': math.nan,
        'sigma0': float(sigma0), 'sigma_effective': math.nan,
        'mu': float(mu), 'min_groups': int(min_groups),
        'huber_delta': delta, 'n_groups_available': 0, 'n_groups_used': 0,
        'n_extrema': 0, 'n_groups_m_lt_2': 0,
        'objective': math.nan, 'fit_rmse_standardized': math.nan,
        'n_downweighted_extrema': 0, 'n_quadratic_extrema': 0,
    }

    def unavailable(status, message):
        info['status'], info['message'] = status, message
        return result, info

    if not enabled:
        return unavailable('disabled', 'Empirical calibration is disabled.')
    if not math.isfinite(float(mu)):
        return unavailable('invalid_mu', 'A finite fixed center is required.')
    if not math.isfinite(float(sigma0)) or sigma0 <= 0:
        return unavailable('invalid_sigma', 'The baseline sigma must be positive and finite.')

    vals = {name: pd.to_numeric(result[name], errors='coerce').to_numpy(dtype=float)
            for name in required}
    n, m = vals['n'], vals['m_n']
    low, high = vals['observed_min'], vals['observed_max']
    valid_n = np.isfinite(n) & (n > 0) & (n == np.floor(n))
    valid_m = np.isfinite(m) & (m >= 2) & (m == np.floor(m))
    eligible = (valid_n & valid_m & np.isfinite(low) & np.isfinite(high)
                & (low <= high))
    result['fit_eligible'] = eligible
    info['n_groups_m_lt_2'] = int(np.count_nonzero(np.isfinite(m) & (m < 2)))
    n_valid = n[eligible]
    info['n_groups_available'] = int(len(np.unique(n_valid)))
    if len(np.unique(n_valid)) != len(n_valid):
        return unavailable('duplicate_n', 'Summary must contain one row per distinct n.')
    if info['n_groups_available'] < min_groups:
        return unavailable('too_few_groups',
                           f'Need at least {min_groups} eligible distinct n groups.')

    # SciPy is needed only after enabled/identifiability checks have passed.
    try:
        from scipy.optimize import brentq
    except ModuleNotFoundError as exc:
        return unavailable('scipy_unavailable', f'SciPy is required for calibration: {exc}')
    idx = np.flatnonzero(eligible)
    try:
        moments = np.array([expected_normal_max_moments(int(v)) for v in m[idx]])
    except (ArithmeticError, ValueError, ModuleNotFoundError) as exc:
        return unavailable('moments_failed', str(exc))
    a, max_sd = moments[:, 0], np.sqrt(moments[:, 1])
    dp = np.sqrt(n[idx]) * (high[idx] - mu) / sigma0
    dm = np.sqrt(n[idx]) * (mu - low[idx]) / sigma0
    for column, values in [('a_m', a), ('max_sd', max_sd),
                           ('d_plus', dp), ('d_minus', dm)]:
        result.iloc[idx, result.columns.get_loc(column)] = values
    x = np.repeat(a / max_sd, 2)
    y = np.column_stack((dp / max_sd, dm / max_sd)).ravel()
    if not np.isfinite(x).all() or not np.isfinite(y).all():
        return unavailable('nonfinite_standardized_data', 'Standardized extrema are not finite.')
    denominator = float(np.dot(x, x))
    k_wls_raw = float(np.dot(x, y) / denominator)
    info['kappa_wls_unconstrained'] = k_wls_raw
    info['kappa_wls'] = max(0.0, k_wls_raw)

    def score(kappa):
        return float(np.dot(x, np.clip(y - kappa * x, -delta, delta)))

    def flat_interval(kappa):
        """Detect an unidentifiable interval where every Huber term is linear."""
        residual = y - kappa * x
        tol = 1e-9 * max(1.0, delta)
        if np.any(np.abs(residual) < delta - tol):
            return None
        positive = residual >= delta - tol
        negative = residual <= -delta + tol
        lower = max(0.0, float(np.max((y[negative] + delta) / x[negative]))
                    if negative.any() else 0.0)
        upper = (float(np.min((y[positive] - delta) / x[positive]))
                 if positive.any() else math.inf)
        if upper - lower > 1e-9 * max(1.0, abs(kappa)):
            return lower, upper
        return None

    if method == 'wls':
        kappa = info['kappa_wls']
    else:
        s0 = score(0.0)
        score_tol = 1e-12 * max(1.0, delta * float(x.sum()))
        if abs(s0) <= score_tol and flat_interval(0.0) is not None:
            info['flat_kappa_interval'] = flat_interval(0.0)
            return unavailable('flat_objective', 'Huber loss does not identify a unique scale.')
        if s0 <= 0.0:
            return unavailable('boundary_zero', 'The constrained optimum is zero; no positive scale is available.')
        upper = max(1.0, float(np.max(y / x)), info['kappa_wls']) * 1.01
        try:
            kappa = float(brentq(score, 0.0, upper, xtol=1e-12, rtol=1e-11))
        except (ValueError, RuntimeError, OverflowError) as exc:
            return unavailable('optimizer_failed', str(exc))
        flat = flat_interval(kappa)
        if flat is not None:
            info['flat_kappa_interval'] = flat
            return unavailable('flat_objective', 'Huber loss does not identify a unique scale.')
    sigma_effective = float(kappa * sigma0)
    if not (math.isfinite(kappa) and kappa > 0.0
            and math.isfinite(sigma_effective) and sigma_effective > 0.0):
        return unavailable('boundary_zero', 'A positive effective scale was not identified.')

    residual = y - kappa * x
    absolute = np.abs(residual)
    weights = np.minimum(1.0, delta / np.maximum(absolute, np.finfo(float).tiny))
    huber_loss = np.where(absolute <= delta, 0.5 * residual**2,
                          delta * (absolute - 0.5 * delta))
    fitted_half = sigma_effective * a / np.sqrt(n[idx])
    outputs = {
        'fit_expected_upper': mu + fitted_half,
        'fit_expected_lower': mu - fitted_half,
        'fit_resid_plus': residual[0::2], 'fit_resid_minus': residual[1::2],
        'huber_weight_plus': weights[0::2], 'huber_weight_minus': weights[1::2],
    }
    for column, values in outputs.items():
        result.iloc[idx, result.columns.get_loc(column)] = values
    result.iloc[idx, result.columns.get_loc('fit_used')] = True
    info.update({
        'status': 'ok', 'message': 'One effective scale calibrated with the center fixed.',
        'kappa': float(kappa), 'sigma_effective': sigma_effective,
        'n_groups_used': len(idx), 'n_extrema': 2 * len(idx),
        'objective': float(np.sum(huber_loss if method == 'huber' else 0.5 * residual**2)),
        'fit_rmse_standardized': float(np.sqrt(np.mean(residual**2))),
        'n_downweighted_extrema': int(np.count_nonzero(absolute > delta)),
        'n_quadratic_extrema': int(np.count_nonzero(absolute < delta)),
    })
    return result, info



def add_empirical_envelope(points, envelope, mu_hat, sigma_hat):
    """기존 열과 후보를 유지하고 경험적 보정 경계/판정을 추가한다."""
    calibrated, fit = empirical_extreme_fit(
        envelope, mu_hat, sigma_hat, enabled=EMPIRICAL_ENVELOPE,
        min_groups=EXTREME_FIT_MIN_GROUPS, huber_delta=EXTREME_FIT_HUBER_DELTA)
    fit['note'] = fit['message']
    available = fit['status'] == 'ok'
    kappa = fit['kappa'] if available else np.nan
    sigma_effective = sigma_hat * kappa
    calibrated['empirical_kappa'] = kappa
    calibrated['sigma_effective'] = sigma_effective
    calibrated['empirical_sigma_hat_n'] = sigma_effective / np.sqrt(calibrated['n'].astype(float))
    calibrated['empirical_half_width'] = calibrated['empirical_sigma_hat_n'] * calibrated['extreme_coefficient'].astype(float)
    calibrated['empirical_lower'] = mu_hat - calibrated['empirical_half_width']
    calibrated['empirical_upper'] = mu_hat + calibrated['empirical_half_width']
    calibrated['empirical_fit_status'] = fit['status']
    calibrated['empirical_fit_note'] = fit['note']
    fields = ['n', 'empirical_kappa', 'sigma_effective', 'empirical_sigma_hat_n',
              'empirical_half_width', 'empirical_lower', 'empirical_upper',
              'empirical_fit_status', 'empirical_fit_note']
    # n의 정수 dtype을 유지해야 빈 후보에서도 안전하게 조인할 수 있다.
    result = points.copy()
    lookup = calibrated.set_index('n')
    for col in fields[1:]:
        result[col] = result['n'].map(lookup[col])
    result['empirical_outside'] = pd.Series(pd.NA, index=result.index, dtype='boolean')
    result['empirical_direction'] = '보정 없음'
    result['envelope_comparison'] = '보정 없음'
    if available:
        scale = result[['sample_mean', 'empirical_lower', 'empirical_upper']].abs().max(axis=1)
        tolerance = 32 * np.finfo(float).eps * scale
        above = result['sample_mean'] > result['empirical_upper'] + tolerance
        below = result['sample_mean'] < result['empirical_lower'] - tolerance
        result['empirical_outside'] = (above | below).astype('boolean')
        result['empirical_direction'] = np.select([above, below], ['위쪽', '아래쪽'], default='경계 안')
        original = result['outside'].fillna(False).to_numpy(bool)
        corrected = result['empirical_outside'].fillna(False).to_numpy(bool)
        result['envelope_comparison'] = np.select(
            [original & corrected, original, corrected],
            ['양쪽 경계 밖', '기존만 경계 밖', '보정만 경계 밖'], default='양쪽 경계 안')
    fit['sigma_effective'] = sigma_effective
    return result, calibrated, fit

def analyze_part_baseline(part_raw, part_id, *, transform=None, use_yeo_johnson=None, lambda_method=None):
    start = time.perf_counter()
    # 이전의 명시적 use_yeo_johnson 호출도 지원한다.
    if use_yeo_johnson is not None:
        if transform is not None or not isinstance(use_yeo_johnson, (bool, np.bool_)):
            raise ValueError('transform과 use_yeo_johnson을 동시에 지정할 수 없으며, 후자는 bool이어야 합니다.')
        selected_transform = 'yeo_johnson' if use_yeo_johnson else 'raw'
    else:
        selected_transform = Y_TRANSFORM if transform is None else transform
    if selected_transform not in ('raw', 'yeo_johnson', 'rank_normal'):
        raise ValueError("transform은 'raw', 'yeo_johnson', 'rank_normal' 중 하나여야 합니다.")
    enabled = selected_transform != 'raw' 
    wf, data, quality = prepare_reference_part(part_raw, part_id)
    if selected_transform == 'rank_normal':
        analysis_y, lambda_info = rank_normal_part_y(wf['raw_y'].to_numpy())
        fitted_lambda = None
    else:
        analysis_y, fitted_lambda, lambda_info = transform_part_y(
            wf['raw_y'].to_numpy(), enabled, method=lambda_method, return_info=True)
    if lambda_info['status'] == 'fallback':
        print(f"  part {part_id}: 강건 λ → MLE 전환: {lambda_info['note']}", flush=True)
    wf['analysis_y'] = analysis_y
    transform = 'yeo-johnson' if selected_transform == 'yeo_johnson' else selected_transform
    if enabled:
        # data['Y']는 그대로 raw: 변환을 켜도 깊이 3+의 후보 선별이 바뀌지 않는다.
        data['analysis_Y'] = analysis_y
    if quality['multiple_y']:
        print(f"  raw y가 여러 개인 웨이퍼 {quality['multiple_y']:,}장: 원본과 같이 첫 값 사용")
    if SHOW_PROGRESS:
        mode = ('Rank-normal (평균 순위)' if transform == 'rank_normal' else
                f"Yeo–Johnson {lambda_info['used']} λ={fitted_lambda:.6g}" if enabled else 'raw (변환 OFF)')
        print(f"  {mode} · 웨이퍼 {data['N']:,}장 / STEP {len(data['steps']):,}개: raw 후보 생성 시작", flush=True)
    if transform == 'rank_normal' and lambda_info['note'] and SHOW_PROGRESS:
        print(f"  {lambda_info['note']}", flush=True)
    reference = analyze(data, min_n=MIN_N, max_depth=MAX_DEPTH, full_depth=2)
    sign = 1 if HIGHER_IS_WORSE else -1
    target_step = (re.sub(STEP_SUFFIX, '', str(STEP)) if STEP_SUFFIX else str(STEP)) if STEP is not None else None
    rows = []
    for c in reference['combos']:
        step = data['steps'][c['step']]
        if target_step is not None and step['name'] != target_step:
            continue
        path = ' → '.join(f"{step['seqs'][q]['name']}: {step['seqs'][q]['units'][u]}" for q, u in c['items'])
        raw_mean = sign * c['mean']
        rows.append({'part_id': part_id, 'step': step['name'], 'depth': c['k'],
                     'path': path, 'n': c['n'], 'raw_mean': raw_mean,
                     'sample_mean': c['analysis_mean'] if enabled else raw_mean, 'combo_key': c['key']})
    points = pd.DataFrame(rows, columns=POINT_COLUMNS + ['raw_mean', 'combo_key'])
    mu_hat_raw = float(wf['raw_y'].mean())
    sigma_hat_raw = float(wf['raw_y'].std(ddof=1))
    mu_hat = float(wf['analysis_y'].mean())
    sigma_hat = float(wf['analysis_y'].std(ddof=1))
    points, envelope = mean_envelope(points, mu_hat, sigma_hat, sigma_hat_raw=sigma_hat_raw)
    # 같은 조합에서 raw 경계와 분석 경계의 판정을 비교한다. 후보 생성은 재실행하지 않는다.
    if enabled:
        raw_points = points[['n', 'raw_mean']].rename(columns={'raw_mean': 'sample_mean'})
        raw_result, raw_envelope = mean_envelope(raw_points, mu_hat_raw, sigma_hat_raw,
                                                sigma_hat_raw=sigma_hat_raw)
    else:
        raw_result, raw_envelope = points, envelope.copy()
    for name in ('lower', 'upper', 'outside', 'direction'):
        points[f'raw_{name}'] = raw_result[name].to_numpy()
    active_out = points['outside'].fillna(False).to_numpy(bool)
    raw_out = points['raw_outside'].fillna(False).to_numpy(bool)
    points['detection_comparison'] = np.select(
        [raw_out & active_out, raw_out, active_out],
        ['양쪽 경계 밖', 'raw만 경계 밖', '분석값만 경계 밖'], default='양쪽 경계 안')
    points['raw_mean_difference'] = points['raw_mean'] - mu_hat_raw
    points['analysis_mean_difference'] = points['sample_mean'] - mu_hat
    points, envelope, extreme_fit = add_empirical_envelope(points, envelope, mu_hat, sigma_hat)
    if SHOW_PROGRESS:
        if extreme_fit['status'] == 'ok':
            print(f"  극값 보정: κ={extreme_fit['kappa']:.4g} / sigma_eff={extreme_fit['sigma_effective']:.5g}", flush=True)
        else:
            print(f"  극값 보정 {extreme_fit['status']}: {extreme_fit['note']}", flush=True)
    raw_envelope['transform'] = 'raw'
    raw_envelope['transform_lambda'] = None
    raw_envelope['mu_hat_raw'] = mu_hat_raw
    for table in (points, envelope):
        table['transform'] = transform
        table['transform_lambda'] = fitted_lambda
        table['mu_hat_raw'] = mu_hat_raw
        table['transform_fit_method'] = lambda_info['used']
        table['transform_fit_status'] = lambda_info['status']
        table['transform_fit_note'] = lambda_info['note']
        is_yj = transform == 'yeo-johnson'
        table['lambda_method_requested'] = lambda_info['requested'] if is_yj else None
        table['lambda_method_used'] = lambda_info['used'] if is_yj else None
        table['lambda_fit_status'] = lambda_info['status'] if is_yj else 'not_applicable'
        table['lambda_fit_note'] = lambda_info['note'] if is_yj else ''
        table['lambda_fit_n_used'] = lambda_info['n_used_last'] if is_yj else None
        table['rank_max_tie_fraction'] = lambda_info.get('max_tie_fraction', np.nan)
        table['rank_n_unique'] = lambda_info.get('n_unique', np.nan)
    return {'_history_data': data, '_history_reference': reference,
            'part_id': part_id, 'wafer_values': wf, 'mu_hat': mu_hat, 'mu_hat_raw': mu_hat_raw,
            'sigma_hat': sigma_hat, 'sigma_hat_raw': sigma_hat_raw, 'points': points,
            'transform': transform, 'transform_lambda': fitted_lambda,
            'transform_fit': lambda_info, 'lambda_fit': lambda_info,
            'envelope': envelope, 'raw_envelope': raw_envelope,
            'extreme_fit': extreme_fit, 'extreme_fit_diagnostics': envelope.copy(), 'quality': quality, 'depth_count': reference['depth_count'],
            'search_space': reference['search_space'], 'seconds': time.perf_counter() - start}


"""Process-history features and sparse membership without changing candidates.

Inputs are the canonical arrays returned by prepare_reference_part; all rework,
wafer eligibility, and candidate selection remain owned by the original code.
"""
from collections import defaultdict
import math
import time
import numpy as np


def _history_array(data, step_index):
    """Validate the positional wafer alignment supplied by the base analyzer."""
    n = int(data['N'])
    array = np.asarray(data['assign'][step_index])
    seqs = data['steps'][step_index]['seqs']
    if array.ndim != 2 or array.shape != (len(seqs), n):
        raise ValueError('History assignment shape does not match step/order/wafer counts')
    if not np.issubdtype(array.dtype, np.integer):
        raise ValueError('History assignments must contain integer unit codes')
    return array


def build_history_codes(data, *, min_coverage=0.5, missing=65535):
    """Return (codes, weights, positions) in the original wafer-column order.

    codes has shape (N, P), dtype int32, and missing=-1. Retain a position only
    when at least two unit choices occur and its coverage reaches min_coverage.
    Every retained STEP/Order position receives the same fixed weight 1/P.
    `positions` describes retained AND excluded positions; history_column=-1
    denotes exclusion. No outcome values or candidate scores enter selection.
    Rework is not re-resolved: assignments are already canonical first/last.
    """
    import pandas as pd
    if not math.isfinite(float(min_coverage)) or not 0 <= min_coverage <= 1:
        raise ValueError('min_coverage must be between zero and one')
    n = int(data['N'])
    if n < 1 or len(data['assign']) != len(data['steps']):
        raise ValueError('History requires at least one wafer and aligned steps/assignments')
    columns, diagnostics = [], []
    for s, step in enumerate(data['steps']):
        array = _history_array(data, s)
        for q, seq in enumerate(step['seqs']):
            values = array[q]
            observed = (values != missing) & (values != -1)
            if np.any(values[observed] < 0) or np.any(values[observed] >= len(seq['units'])):
                raise ValueError(f'Invalid unit code at STEP {s}, Order {q}')
            count = int(observed.sum())
            unique_count = int(np.unique(values[observed]).size)
            coverage = count / n
            kept = unique_count >= 2 and coverage >= min_coverage
            reason = ('kept' if kept else 'fewer_than_two_units' if unique_count < 2
                      else 'coverage_below_threshold')
            position = len(columns) if kept else -1
            if kept:
                columns.append((s, q))
            diagnostics.append({
                'step_index': s, 'order_index': q,
                'step': str(step['name']), 'order': str(seq.get('name', q)),
                'position_label': f"{step['name']} / {seq.get('name', q)}",
                'n_observed': count, 'n_wafers': n, 'coverage': coverage,
                'n_units_observed': unique_count, 'kept': kept,
                'history_column': position, 'reason': reason,
            })
    # Allocate the final matrix once; do not retain another N×P column cache.
    codes = np.full((n, len(columns)), -1, dtype=np.int32)
    for column_index, (s, q) in enumerate(columns):
        values = np.asarray(data['assign'][s])[q]
        observed = (values != missing) & (values != -1)
        codes[observed, column_index] = values[observed].astype(np.int32, copy=False)
    weights = np.full(codes.shape[1], 1 / codes.shape[1], dtype=float) if codes.shape[1] else np.empty(0)
    names = ['step_index', 'order_index', 'step', 'order', 'position_label',
             'n_observed', 'n_wafers', 'coverage', 'n_units_observed', 'kept',
             'history_column', 'reason']
    positions = pd.DataFrame(diagnostics, columns=names)
    positions['weight'] = np.where(positions['kept'], 1 / codes.shape[1] if codes.shape[1] else 0.0, 0.0)
    positions.attrs['n_wafers_without_retained_history'] = int(np.count_nonzero(~(codes >= 0).any(axis=1)))
    positions.attrs['min_coverage'] = float(min_coverage)
    return codes, weights, positions


def build_wafer_index_map(wafer_frame):
    """Export the CSR column index and stable wafer identifiers, without sorting.

    wafer_frame is report['wafer_values'] or the wf returned by preparation.
    Retains wafer order exactly; never infer it from sorted IDs at export time.
    """
    import pandas as pd
    names = ['root_lot_id', 'wafer_id']
    if not isinstance(wafer_frame.index, pd.MultiIndex) or any(name not in wafer_frame.index.names for name in names):
        raise ValueError('Wafer frame needs root_lot_id/wafer_id MultiIndex levels')
    result = pd.DataFrame({name: wafer_frame.index.get_level_values(name).to_numpy(copy=True) for name in names})
    if result.duplicated(names).any():
        raise ValueError('Wafer identifiers must be unique within the part')
    result.insert(0, 'wafer_index', np.arange(len(result), dtype=np.int64))
    return result


def build_combo_membership(data, selected_combos, *, missing=65535,
                           progress=None, max_nnz=None):
    """Return (CSR membership, diagnostics), preserving selected_combos order.

    Rows are displayed combos, columns are canonical wafer indices, entries
    are float64 ones. Every row sum is checked against its original c['n'].
    Combos must be the original analyzer dictionaries (step, items, n, key).

    Groups sharing (STEP, selected Orders) are partitioned together: one wafer
    scan and one stable key sort per group, then only the requested unit cells
    are copied. No combo-by-wafer dense matrix or per-combo wafer scan is used.
    A single partition cache is live at once. For enormous mixed-radix spaces,
    lexicographic structured keys avoid int64 overflow without enumerating cells.

    max_nnz optionally limits the requested output size before allocation.
    progress is an optional callable receiving readable progress messages.
    """
    from scipy.sparse import csr_matrix
    start = time.perf_counter()
    n = int(data['N'])
    combos = list(selected_combos)
    if n < 1:
        raise ValueError('Membership requires a positive wafer count')
    grouped = defaultdict(list)
    expected = np.empty(len(combos), dtype=np.int64)
    seen = set()
    for row, combo in enumerate(combos):
        s = int(combo['step'])
        if s < 0 or s >= len(data['steps']):
            raise ValueError('Combo references an invalid STEP')
        raw_items = tuple(combo['items'])
        items = tuple(sorted((int(q), int(u)) for q, u in raw_items))
        if not items or len({q for q, _ in items}) != len(items):
            raise ValueError('Each combo must contain distinct Order indices')
        seqs = data['steps'][s]['seqs']
        for q, u in items:
            if q < 0 or q >= len(seqs) or u < 0 or u >= len(seqs[q]['units']):
                raise ValueError('Combo references an invalid Order/unit')
        identity = (s, items)
        if identity in seen:
            raise ValueError('Selected combinations must be unique')
        seen.add(identity)
        count = int(combo['n'])
        if count != combo['n'] or count < 1 or count > n:
            raise ValueError('Combo wafer counts must be positive integers no greater than N')
        expected[row] = count
        qs = tuple(q for q, _ in items)
        us = tuple(u for _, u in items)
        grouped[(s, qs)].append((row, us))
    total = sum(map(int, expected))
    if max_nnz is not None:
        if isinstance(max_nnz, bool) or int(max_nnz) != max_nnz or max_nnz < 0:
            raise ValueError('max_nnz must be a nonnegative integer or None')
        if total > max_nnz:
            raise MemoryError(f'Membership requires {total:,} entries, above max_nnz={max_nnz:,}')
    if total > np.iinfo(np.intp).max:
        raise MemoryError('Membership has too many entries for this platform')
    indptr = np.empty(len(combos) + 1, dtype=np.int64)
    indptr[0] = 0
    np.cumsum(expected, out=indptr[1:])
    index_dtype = np.int32 if n <= np.iinfo(np.int32).max else np.int64
    indices = np.empty(total, dtype=index_dtype)
    last_progress = start
    for group_index, ((s, qs), targets) in enumerate(grouped.items(), 1):
        array = _history_array(data, s)[list(qs)]
        observed = np.all((array != missing) & (array != -1), axis=0)
        wafer_indices = np.flatnonzero(observed)
        values = array[:, observed]
        dims = [len(data['steps'][s]['seqs'][q]['units']) for q in qs]
        if np.any(values < 0) or any(np.any(values[i] >= d) for i, d in enumerate(dims)):
            raise ValueError('History assignments contain invalid observed unit codes')
        if math.prod(dims) <= np.iinfo(np.int64).max:
            keys = np.zeros(len(wafer_indices), dtype=np.int64)
            for i, d in enumerate(dims):
                keys = keys * d + values[i]
            order = np.argsort(keys, kind='stable')
            sorted_keys = keys[order]
            target_keys = np.zeros(len(targets), dtype=np.int64)
            for j, (_, units) in enumerate(targets):
                value = 0
                for d, u in zip(dims, units):
                    value = value * d + u
                target_keys[j] = value
        else:
            dtype = np.dtype([(f'q{i}', np.int64) for i in range(len(qs))])
            keys = np.empty(len(wafer_indices), dtype=dtype)
            target_keys = np.empty(len(targets), dtype=dtype)
            for i in range(len(qs)):
                keys[f'q{i}'] = values[i]
                target_keys[f'q{i}'] = [units[i] for _, units in targets]
            order = np.argsort(keys, kind='stable')
            sorted_keys = keys[order]
        starts = np.searchsorted(sorted_keys, target_keys, side='left')
        stops = np.searchsorted(sorted_keys, target_keys, side='right')
        grouped_wafers = wafer_indices[order]
        for (row, _), lo, hi in zip(targets, starts, stops):
            actual = int(hi - lo)
            if actual != expected[row]:
                label = combos[row].get('key', row)
                raise ValueError(f'Combo {label}: membership has {actual} wafers, expected {expected[row]}')
            indices[indptr[row]:indptr[row + 1]] = grouped_wafers[lo:hi]
        now = time.perf_counter()
        if progress is not None and (now - last_progress >= 5 or group_index == len(grouped)):
            progress(f'소속 행렬: Order 묶음 {group_index:,}/{len(grouped):,} · {len(combos):,}개 조합')
            last_progress = now
    membership = csr_matrix((np.ones(total, dtype=float), indices, indptr),
                            shape=(len(combos), n), copy=False)
    membership.sort_indices()
    if not np.array_equal(np.asarray(membership.sum(axis=1)).ravel(), expected):
        raise AssertionError('Membership row counts differ from original candidate counts')
    info = {'n_combos': len(combos), 'n_wafers': n, 'nnz': total,
            'n_order_groups': len(grouped),
            'csr_bytes': int(membership.data.nbytes + membership.indices.nbytes + membership.indptr.nbytes),
            'seconds': time.perf_counter() - start}
    return membership, info


"""Bounded-memory history-kernel covariance fitting and mean contrasts.

One common intercept is profiled by Gaussian REML.  History similarity is a
covariance MODEL, not an observed correlation.  Unmodelled process mean effects
can be absorbed by this model.  Nyström plus a diagonal remainder approximates
the kernel; diagnostics report that approximation explicitly.
"""

import math
import numpy as np
import pandas as pd


def _paired_history_distance(codes, left, right, weights):
    """Fixed-coordinate weighted Hamming; unknown categories are wafer-unique."""
    left, right = np.asarray(left, dtype=int), np.asarray(right, dtype=int)
    distance = np.zeros(np.broadcast_shapes(left.shape, right.shape), dtype=float)
    same_wafer = left == right
    for j, weight in enumerate(weights):
        if weight == 0:
            continue
        a, b = codes[left, j], codes[right, j]
        match = ((a >= 0) & (b >= 0) & (a == b)) | same_wafer
        distance += weight * (~match)
    return distance


def _history_kernel(codes, left, right, weights, ell):
    left, right = np.asarray(left, dtype=int), np.asarray(right, dtype=int)
    distance = _paired_history_distance(codes, left[:, None], right[None, :], weights)
    return np.exp(-distance / ell)


def _distinct_pairs(pool, size, rng):
    pool = np.asarray(pool, dtype=int)
    if len(pool) < 2:
        return np.array([], dtype=int), np.array([], dtype=int)
    i = rng.integers(0, len(pool), size=size)
    j = rng.integers(0, len(pool) - 1, size=size)
    j += j >= i
    return pool[i], pool[j]


def _unique_landmarks(codes, candidates, limit):
    seen, chosen = set(), []
    for index in candidates:
        row = codes[index]
        # A missing category belongs to this wafer alone, including for dedupe.
        signature = (row.tobytes(), int(index) if np.any(row < 0) else None)
        if signature not in seen:
            chosen.append(int(index))
            seen.add(signature)
            if len(chosen) >= limit:
                break
    return np.asarray(chosen, dtype=int)


def _nystrom_basis(codes, rows, landmarks, weights, ell, basis, chunk=1024):
    rows = np.asarray(rows, dtype=int)
    output = np.empty((len(rows), basis.shape[1]), dtype=float)
    max_overshoot, clipped = 0.0, 0
    for start in range(0, len(rows), chunk):
        stop = min(start + chunk, len(rows))
        factor = _history_kernel(codes, rows[start:stop], landmarks, weights, ell) @ basis
        norm2 = np.einsum('ij,ij->i', factor, factor)
        above = norm2 > 1.0
        if above.any():
            max_overshoot = max(max_overshoot, float((norm2[above] - 1.0).max()))
            clipped += int(above.sum())
            factor[above] /= np.sqrt(norm2[above])[:, None]
        output[start:stop] = factor
    remainder = np.maximum(0.0, 1.0 - np.einsum('ij,ij->i', output, output))
    return output, remainder, max_overshoot, clipped


def _profile_reml(y, factor, diagonal):
    """Profile mean and overall scale for V0 = FF' + diag(diagonal)."""
    from scipy.linalg import cho_factor, cho_solve

    y = np.asarray(y, dtype=float)
    diagonal = np.asarray(diagonal, dtype=float)
    n = len(y)
    if n < 2 or np.any(diagonal <= 0) or not np.isfinite(diagonal).all():
        raise ValueError('REML needs >=2 observations and a positive diagonal.')
    center = float(y.mean())
    response = y - center
    rhs = np.column_stack((np.ones(n), response))
    inverse_d = 1.0 / diagonal
    inverse_rhs = inverse_d[:, None] * rhs
    logdet = float(np.log(diagonal).sum())
    if factor.shape[1]:
        weighted_factor = inverse_d[:, None] * factor
        small = np.eye(factor.shape[1]) + factor.T @ weighted_factor
        chol = cho_factor(small, lower=True, check_finite=False)
        projected = factor.T @ inverse_rhs
        inverse_rhs -= weighted_factor @ cho_solve(chol, projected, check_finite=False)
        logdet += 2.0 * float(np.log(np.diag(chol[0])).sum())
    information = float(inverse_rhs[:, 0].sum())
    cross = float(inverse_rhs[:, 1].sum())
    quadratic = float(response @ inverse_rhs[:, 1] - cross * cross / information)
    sigma2 = quadratic / (n - 1)
    if information <= 0 or not math.isfinite(sigma2) or sigma2 <= 0:
        raise ArithmeticError('REML scale/intercept information is not positive.')
    negative_loglik = 0.5 * (
        (n - 1) * (math.log(2.0 * math.pi) + 1.0 + math.log(sigma2))
        + logdet + math.log(information)
    )
    return {'sigma2': float(sigma2), 'mu_gls': center + cross / information,
            'negative_reml_loglik': float(negative_loglik),
            'reml_loglik': float(-negative_loglik)}


def fit_history_covariance(codes, y, *, weights=None, max_landmarks=64,
                           max_fit=1200, seed=42, enabled=True, progress=None):
    """Fit one part's low-rank Gaussian history covariance.

    codes: N x P integer categories; negative categories are wafer-specific
    missing states, so two missing observations never match one another.
    weights: fixed nonnegative coordinate weights, normalized once globally.

    Model contract:
        Cov(y) = sigma2 * (F @ F.T + diag(diagonal)).
    F already includes sqrt(rho); diagonal preserves a unit marginal diagonal.
    No N x N matrix is constructed.  max_fit and max_landmarks are the explicit
    memory/work bounds; neither is silently clamped.  progress, when provided,
    is called with a short string before each length-scale candidate.
    OFF/unsupported states return an explicitly labelled iid reference when
    its variance is available, NOT a claimed history correction.
    """
    codes = np.asarray(codes)
    y = np.asarray(y, dtype=float)
    if codes.ndim != 2 or y.ndim != 1 or len(y) != codes.shape[0]:
        raise ValueError('codes must be N x P and y must be length N.')
    if not np.issubdtype(codes.dtype, np.integer):
        raise ValueError('history codes must have an integer dtype.')
    n, p = codes.shape
    if weights is None:
        weights = np.ones(p, dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)
    if weights.shape != (p,) or not np.isfinite(weights).all() or np.any(weights < 0):
        raise ValueError('weights must be finite nonnegative values, one per coordinate.')
    if (isinstance(max_landmarks, bool) or isinstance(max_fit, bool)
            or int(max_landmarks) != max_landmarks or int(max_fit) != max_fit
            or max_landmarks < 1 or max_fit < 2):
        raise ValueError('Integer max_landmarks >=1 and max_fit >=2 are required.')
    max_landmarks, max_fit = int(max_landmarks), int(max_fit)
    if progress is not None and not callable(progress):
        raise ValueError('progress must be callable or None.')
    reference_variance = (float(np.var(y, ddof=1))
                          if n >= 2 and np.isfinite(y).all() else math.nan)
    diagnostics = {
        'status': 'unavailable', 'message': '', 'enabled': bool(enabled),
        'history_correction_applied': False, 'is_available': False,
        'n_wafers': n, 'n_coordinates': p, 'fit_n': 0,
        'n_landmarks': 0, 'rank': 0, 'rho': 0.0, 'ell': math.nan,
        'reference_sigma2': reference_variance,
        'covariance_sigma2': reference_variance,
        'covariance_sigma': math.sqrt(reference_variance) if math.isfinite(reference_variance) and reference_variance >= 0 else math.nan,
        'missing_fraction': float(np.mean(codes < 0)) if codes.size else math.nan,
        'n_wafers_without_observed_history': int(np.sum(np.all(codes < 0, axis=1))) if p else n,
        'approx_kernel_rmse': math.nan, 'approx_kernel_max_error': math.nan,
        'approx_kernel_mean_absolute_error': math.nan,
        'approx_correlation_rmse': math.nan, 'approx_pairs': 0,
        'nystrom_residual_mean': math.nan, 'nystrom_residual_max': math.nan,
        'nystrom_residual_q95': math.nan, 'approximation_warning': False,
        'fit_candidate_failures': 0, 'interpretation': 'fitted_covariance_model_not_measured_pairwise_correlations',
        'warnings': [],
    }
    model = {
        'F': np.empty((n, 0), dtype=float), 'diagonal': np.ones(n, dtype=float),
        'sigma2': reference_variance, 'scale': reference_variance,
        'rho': 0.0, 'ell': math.nan, 'mu_gls': float(np.mean(y)) if n else math.nan,
        'n_wafers': n, 'diagnostics': diagnostics, 'fit_table': pd.DataFrame(),
        'fit_indices': np.array([], dtype=int), 'landmark_indices': np.array([], dtype=int),
        'weights': weights.copy(),
    }

    def reference(status, message):
        diagnostics.update(status=status, message=message,
                           is_available=bool(math.isfinite(reference_variance) and reference_variance > 0))
        return model

    if not enabled:
        return reference('disabled_iid_reference', 'History fitting disabled; covariance is the iid reference.')
    if n < 4:
        return reference('insufficient_wafers_iid_reference', 'At least four wafers are required to attempt covariance fitting.')
    if not np.isfinite(y).all():
        return reference('nonfinite_y_unavailable', 'Responses must all be finite; no fitted correction is available.')
    if not math.isfinite(reference_variance) or reference_variance <= 0:
        return reference('zero_variance_unavailable', 'Response variance is zero; a positive covariance scale cannot be fitted.')
    if p == 0 or weights.sum() <= 0:
        return reference('no_features_iid_reference', 'No weighted varying history coordinates are available.')
    if not np.any(codes[:, weights > 0] >= 0):
        return reference('no_observed_history_iid_reference', 'All weighted history entries are missing.')
    weights = weights / weights.sum()
    model['weights'] = weights.copy()
    try:
        from scipy.linalg import eigh
    except ModuleNotFoundError as exc:
        return reference('scipy_unavailable_iid_reference', f'SciPy is unavailable: {exc}')
    rng = np.random.default_rng(seed)
    fit_indices = np.sort(rng.choice(n, size=min(n, max_fit), replace=False))
    model['fit_indices'] = fit_indices
    diagnostics['fit_n'] = len(fit_indices)
    pair_i, pair_j = _distinct_pairs(fit_indices, min(4096, max(256, len(fit_indices) * 4)), rng)
    distances = _paired_history_distance(codes, pair_i, pair_j, weights)
    diagnostics['distance_q10'], diagnostics['distance_q50'], diagnostics['distance_q90'] = (
        float(v) for v in np.quantile(distances, [0.1, 0.5, 0.9]))
    if np.ptp(distances) < 1e-12:
        return reference('constant_sampled_distances_iid_reference',
                         'Sampled off-diagonal history distances are constant; history correlation is not separated from scale/intercept.')
    positive = distances[distances > 1e-12]
    if not len(positive):
        return reference('identical_histories_iid_reference', 'No history distance variation remains after the common intercept.')
    q01, q10, q50, q90 = np.quantile(positive, [0.01, 0.1, 0.5, 0.9])
    # Small recurring bundles can occupy <10% of pairs; keep a local scale in
    # the grid instead of using only median/global pair distances.
    ell_grid = np.unique(np.maximum(1e-6, [0.5 * positive.min(), q01,
                                          0.5 * q10, q50, q90, 2.0 * q90]))
    rho_grid = (0.25, 0.5, 0.75, 0.9, 0.99)
    diagnostics['ell_grid'] = [float(value) for value in ell_grid]
    diagnostics['rho_grid'] = [0.0, *rho_grid]
    landmarks = _unique_landmarks(codes, rng.permutation(fit_indices), min(max_landmarks, len(fit_indices)))
    model['landmark_indices'] = landmarks
    diagnostics['n_landmarks'] = len(landmarks)
    fit_y = y[fit_indices]
    table_rows, bases = [], {}
    iid = _profile_reml(fit_y, np.empty((len(fit_y), 0)), np.ones(len(fit_y)))
    table_rows.append(dict(ell=math.nan, rho=0.0, rank=0, fit_n=len(fit_y),
                           sigma2=iid['sigma2'], mu_gls=iid['mu_gls'],
                           negative_reml_loglik=iid['negative_reml_loglik'],
                           reml_loglik=iid['reml_loglik'], status='ok', selected=False))
    best_index, best_nll = 0, iid['negative_reml_loglik']
    for ell_index, ell in enumerate(ell_grid):
        if progress is not None:
            progress(f'History REML: ell {ell_index + 1}/{len(ell_grid)} = {ell:.5g}; '
                     f'{len(fit_indices)} fit wafers, {len(landmarks)} landmarks')
        try:
            anchor_kernel = _history_kernel(codes, landmarks, landmarks, weights, float(ell))
            eigenvalues, eigenvectors = eigh(anchor_kernel, check_finite=False)
            keep = eigenvalues > max(1e-11, float(eigenvalues[-1]) * 1e-10)
            if not keep.any():
                raise ArithmeticError('Nyström anchor matrix has no retained positive eigenvalue.')
            basis = eigenvectors[:, keep] / np.sqrt(eigenvalues[keep])[None, :]
            f0, remainder, overshoot, _ = _nystrom_basis(codes, fit_indices, landmarks, weights, float(ell), basis)
            bases[float(ell)] = basis
            for rho in rho_grid:
                try:
                    fitted = _profile_reml(fit_y, math.sqrt(rho) * f0, 1.0 - rho + rho * remainder)
                    row = dict(ell=float(ell), rho=rho, rank=f0.shape[1], fit_n=len(fit_y),
                               sigma2=fitted['sigma2'], mu_gls=fitted['mu_gls'],
                               negative_reml_loglik=fitted['negative_reml_loglik'],
                               reml_loglik=fitted['reml_loglik'], status='ok', selected=False)
                    table_rows.append(row)
                    # Prefer the simpler iid candidate in numerically indistinguishable ties.
                    if fitted['negative_reml_loglik'] < best_nll - 1e-8:
                        best_nll, best_index = fitted['negative_reml_loglik'], len(table_rows) - 1
                except (ValueError, ArithmeticError, np.linalg.LinAlgError) as exc:
                    diagnostics['fit_candidate_failures'] += 1
                    table_rows.append(dict(ell=float(ell), rho=rho, rank=f0.shape[1], fit_n=len(fit_y),
                                           status=f'failed: {exc}', selected=False))
        except (ValueError, ArithmeticError, np.linalg.LinAlgError) as exc:
            diagnostics['fit_candidate_failures'] += len(rho_grid)
            table_rows.append(dict(ell=float(ell), rho=math.nan, rank=0, fit_n=len(fit_y),
                                   status=f'kernel_failed: {exc}', selected=False))
    table_rows[best_index]['selected'] = True
    model['fit_table'] = pd.DataFrame(table_rows)
    selected = table_rows[best_index]
    diagnostics['reml_loglik_gain_over_iid'] = float(iid['negative_reml_loglik'] - best_nll)
    if selected['rho'] == 0.0:
        # With an iid model use all available responses for its final scale.
        model['mu_gls'] = float(y.mean())
        if not any(row.get('status') == 'ok' and row.get('rho', 0.0) > 0 for row in table_rows):
            return reference('fit_failed_iid_reference', 'All history candidates failed; returning the explicit iid reference.')
        return reference('iid_selected', 'The grid selected the iid covariance; no history-dependent correction is applied.')

    rho, ell = float(selected['rho']), float(selected['ell'])
    f0, remainder, overshoot, clipped = _nystrom_basis(
        codes, np.arange(n), landmarks, weights, ell, bases[ell])
    model.update(F=math.sqrt(rho) * f0, diagonal=1.0 - rho + rho * remainder,
                 sigma2=float(selected['sigma2']), scale=float(selected['sigma2']),
                 rho=rho, ell=ell, mu_gls=float(selected['mu_gls']))
    diagnostics.update(status='ok', message='History covariance selected by Gaussian REML on a bounded wafer sample.',
                       history_correction_applied=True, is_available=True,
                       rho=rho, ell=ell, rank=f0.shape[1],
                       nystrom_residual_mean=float(remainder.mean()),
                       nystrom_residual_max=float(remainder.max()),
                       nystrom_residual_q95=float(np.quantile(remainder, 0.95)),
                       nystrom_norm_overshoot=float(overshoot), nystrom_clipped_rows=int(clipped),
                       covariance_sigma2=float(selected['sigma2']),
                       covariance_sigma=float(math.sqrt(selected['sigma2'])))
    if rho == max(rho_grid):
        diagnostics['warnings'].append('rho reaches the upper search-grid boundary; its strength is not finely resolved by this grid.')
    if ell == float(ell_grid[0]) or ell == float(ell_grid[-1]):
        side = 'lower' if ell == float(ell_grid[0]) else 'upper'
        diagnostics['warnings'].append(f'ell reaches the {side} search-grid boundary; check sensitivity to the candidate range.')
    nonanchors = np.setdiff1d(np.arange(n), landmarks, assume_unique=False)
    qa_pool = nonanchors if len(nonanchors) >= 2 else np.arange(n)
    qi, qj = _distinct_pairs(qa_pool, min(4096, max(512, n * 2)), rng)
    exact = np.exp(-_paired_history_distance(codes, qi, qj, weights) / ell)
    approximation = np.einsum('ij,ij->i', f0[qi], f0[qj])
    error = approximation - exact
    rmse, maximum = float(np.sqrt(np.mean(error**2))), float(np.max(np.abs(error)))
    diagnostics.update(approx_kernel_rmse=rmse, approx_kernel_max_error=maximum,
                       approx_kernel_mean_absolute_error=float(np.mean(np.abs(error))),
                       approx_correlation_rmse=rho * rmse, approx_pairs=len(qi),
                       approx_pairs_exclude_landmarks=bool(len(nonanchors) >= 2),
                       diagonal_max_error=float(np.max(np.abs(np.einsum('ij,ij->i', model['F'], model['F'])
                                                               + model['diagonal'] - 1.0))))
    # These are reporting heuristics, not statistical guarantees.
    poor = rho * rmse > 0.05 or rho * maximum > 0.25
    diagnostics['approximation_warning'] = bool(poor)
    if poor:
        diagnostics['warnings'].append('Nyström held-out pair error is material; increase landmarks or treat correction as approximate.')
    if remainder.mean() > 0.5:
        diagnostics['warnings'].append('Over half the kernel diagonal remains in the diagonal approximation; unrepresented local dependence may be missed.')
    if len(fit_indices) < n:
        diagnostics['warnings'].append('Kernel hyperparameters and scale were fitted on a wafer subsample.')
    diagnostics['warnings'].append('Similarity-derived covariance can absorb unmodelled process mean effects; it is not causal attribution.')
    return model


def covariance_metrics(model, membership, *, chunk_size=4096):
    """Return exact mean/whole-part-contrast variances for the fitted model.

    membership is a sparse CSR indicator matrix: rows are candidate means and
    columns are the model's wafers.  Duplicate nonzero entries are deduplicated.
    n_eff refers to the uncentered mean and is only a marginal diagnostic; the
    plotted comparison must use variance_contrast when centered on all wafers.
    """
    from scipy.sparse import csr_matrix

    matrix = csr_matrix(membership, dtype=float, copy=True)
    matrix.sum_duplicates()
    matrix.eliminate_zeros()
    if not np.isfinite(matrix.data).all() or np.any(matrix.data < 0):
        raise ValueError('Membership entries must be finite and nonnegative.')
    matrix.data[:] = 1.0
    factor = np.asarray(model['F'], dtype=float)
    diagonal = np.asarray(model['diagonal'], dtype=float)
    sigma2 = float(model['sigma2'])
    n = len(diagonal)
    if factor.ndim != 2 or factor.shape[0] != n or matrix.shape[1] != n:
        raise ValueError('Model and membership wafer dimensions do not match.')
    if np.any(diagonal < 0) or not np.isfinite(diagonal).all() or not np.isfinite(factor).all():
        raise ValueError('Model covariance factor/diagonal is invalid.')
    if chunk_size < 1:
        raise ValueError('chunk_size must be positive.')
    counts = np.asarray(matrix.sum(axis=1)).ravel()
    columns = ['variance_mean', 'variance_contrast', 'se_mean', 'se_contrast',
               'n_eff', 'variance_mean_unit_scale', 'variance_contrast_unit_scale']
    result = pd.DataFrame({'n': counts.astype(int)})
    for column in columns:
        result[column] = np.nan
    result['valid_membership'] = counts > 0
    result['zero_contrast_variance'] = False
    if n == 0 or not math.isfinite(sigma2) or sigma2 < 0:
        return result
    sum_f = factor.sum(axis=0)
    sum_d = float(diagonal.sum())
    for start in range(0, matrix.shape[0], int(chunk_size)):
        stop = min(start + int(chunk_size), matrix.shape[0])
        block = matrix[start:stop]
        ng = counts[start:stop]
        valid = ng > 0
        if not valid.any():
            continue
        sf = np.asarray(block @ factor)
        sd = np.asarray(block @ diagonal).ravel()
        ng_safe = np.where(valid, ng, 1.0)
        mean_factor = sf / ng_safe[:, None]
        mean_variance0 = np.einsum('ij,ij->i', mean_factor, mean_factor) + sd / ng_safe**2
        contrast_factor = mean_factor - sum_f[None, :] / n
        residual_variance = (sd * (1.0 / ng_safe - 1.0 / n)**2
                             + np.maximum(0.0, sum_d - sd) / n**2)
        contrast_variance0 = np.einsum('ij,ij->i', contrast_factor, contrast_factor) + residual_variance
        # A mean over every wafer is precisely the reference mean.
        contrast_variance0[ng == n] = 0.0
        row_index = np.arange(start, stop)[valid]
        values = {
            'variance_mean': sigma2 * mean_variance0,
            'variance_contrast': sigma2 * contrast_variance0,
            'se_mean': np.sqrt(sigma2 * mean_variance0),
            'se_contrast': np.sqrt(sigma2 * contrast_variance0),
            'n_eff': np.divide(1.0, mean_variance0, out=np.full_like(mean_variance0, np.nan),
                               where=mean_variance0 > 0),
            'variance_mean_unit_scale': mean_variance0,
            'variance_contrast_unit_scale': contrast_variance0,
        }
        for key, value in values.items():
            result.loc[row_index, key] = value[valid]
        result.loc[row_index, 'zero_contrast_variance'] = contrast_variance0[valid] == 0.0
    return result


"""Gaussian-model expected absolute extrema for overlapping process combinations."""
import time
import numpy as np
import pandas as pd


def history_expected_extremes(model, membership, n_values, variance_contrast, *,
                             draws=1024, min_draws=128, operation_budget=250_000_000,
                             batch_size=32, seed=42, progress=None):
    """Average simulated Gaussian maxima, never a percentile of observed y.

    A draw is made once per wafer (shared across all combinations). Low-rank
    history effects and the same-data part mean are shared as well. Thus the
    simulation includes both history dependence and overlapping membership.
    Covariance parameters and the observed candidate set are held fixed.
    """
    from scipy import sparse
    if isinstance(draws, bool) or int(draws) != draws or draws < 2:
        raise ValueError('draws must be an integer >= 2')
    if min_draws < 2 or min_draws > draws or operation_budget < 1 or batch_size < 1:
        raise ValueError('Invalid Monte Carlo settings')
    a = sparse.csr_matrix(membership, dtype=float)
    n = np.asarray(n_values, dtype=float)
    v = np.asarray(variance_contrast, dtype=float)
    if n.shape != (a.shape[0],) or v.shape != n.shape:
        raise ValueError('Membership, n, and contrast variance are not aligned')
    if len(n) == 0:
        return pd.DataFrame(columns=['n', 'm_n', 'history_c', 'history_c_mc_se',
                                     'history_mc_draws', 'zero_variance_points']), {
            'status': 'empty', 'draws_requested': int(draws), 'draws_used': 0}
    if np.any(n <= 0) or not np.isfinite(v).all() or np.any(v < 0):
        raise ValueError('Invalid combination sizes or covariance variances')
    np.testing.assert_allclose(np.asarray(a.sum(axis=1)).ravel(), n, atol=0, rtol=0)
    f = np.asarray(model['F'], dtype=float)
    diagonal = np.asarray(model['diagonal'], dtype=float)
    sigma2 = float(model['sigma2'])
    if f.shape[0] != a.shape[1] or diagonal.shape != (a.shape[1],):
        raise ValueError('Covariance model is not aligned with wafer membership')
    if sigma2 <= 0 or not np.isfinite(sigma2) or np.any(diagonal < 0):
        raise ValueError('Covariance model must have finite positive scale')
    unit_se = np.sqrt(v / sigma2)
    valid = unit_se > 1e-12
    # Soft work budget: includes membership, latent-factor, and wafer draw work.
    # Minimum draws take precedence; this is an estimate, not a wall-time cap.
    work_per_draw = max(a.nnz + a.shape[1] * (f.shape[1] + 1) + a.shape[0], 1)
    used = min(int(draws), max(int(min_draws), int(operation_budget // work_per_draw)))
    used = max(int(min_draws), (used // min(32, int(min_draws))) * min(32, int(min_draws)))
    used = min(int(draws), used)
    batch = min(int(batch_size), used,
                max(1, 8_000_000 // max(a.shape[0] + a.shape[1] + f.shape[1], 1)))
    order = np.argsort(n, kind='stable')
    ns, starts, counts = np.unique(n[order], return_index=True, return_counts=True)
    aa = a[order].multiply((1.0 / n[order])[:, None]).tocsr()
    ordered_se = unit_se[order]
    ordered_valid = valid[order]
    sums = np.zeros(len(ns), dtype=float)
    sums2 = np.zeros(len(ns), dtype=float)
    rng = np.random.default_rng(seed)
    last = time.perf_counter()
    for start in range(0, used, batch):
        b = min(batch, used - start)
        # Keep the joint covariance intact: do not draw independently per point.
        eps = rng.standard_normal((a.shape[1], b)) * np.sqrt(diagonal)[:, None]
        if f.shape[1]:
            eps += f @ rng.standard_normal((f.shape[1], b))
        z = aa @ eps
        z -= eps.mean(axis=0)[None, :]
        z[ordered_valid] /= ordered_se[ordered_valid, None]
        z[~ordered_valid] = 0.0
        maxima = np.maximum.reduceat(np.abs(z), starts, axis=0)
        sums += maxima.sum(axis=1)
        sums2 += np.einsum('ij,ij->i', maxima, maxima)
        if progress is not None and time.perf_counter() - last >= 5:
            progress(f'이론 Gaussian 기대 극값: {start + b:,}/{used:,}회')
            last = time.perf_counter()
    mean = sums / used
    sample_variance = np.maximum((sums2 - sums * sums / used) / (used - 1), 0.0)
    se = np.sqrt(sample_variance / used)
    # E|Z| is analytic when only one stochastic point is present.
    stochastic_count = np.add.reduceat(ordered_valid.astype(int), starts)
    one = stochastic_count == 1
    none = stochastic_count == 0
    mean[one], se[one] = np.sqrt(2 / np.pi), 0.0
    mean[none], se[none] = 0.0, 0.0
    result = pd.DataFrame({'n': ns.astype(int), 'm_n': counts.astype(int),
                           'history_c': mean, 'history_c_mc_se': se,
                           'history_mc_draws': used,
                           'zero_variance_points': counts - stochastic_count})
    info = {'status': 'ok', 'draws_requested': int(draws), 'draws_used': used,
            'operation_budget': int(operation_budget), 'batch_size': batch, 'seed': int(seed),
            'estimated_work_per_draw': int(work_per_draw),
            'estimated_total_work': int(work_per_draw * used),
            'minimum_draws_exceed_budget': bool(work_per_draw * used > operation_budget),
            'budget_reduced': used < draws, 'max_mc_se': float(se.max()),
            'statistic': 'E[max |standardized combination mean - part mean|]',
            'interpretation': 'Gaussian-model expectation; not a quantile or false-positive guarantee'}
    return result, info


def attach_history_result(report, data, reference):
    """Preserve every original candidate and mean; add history-model diagnostics."""
    part_id = report['part_id']
    progress = (lambda msg: print(f'  part {part_id}: {msg}', flush=True)) if SHOW_PROGRESS else None
    points = report['points'].copy()
    selected = [reference['by_key'][key] for key in points['combo_key']]
    membership, membership_info = build_combo_membership(data, selected, progress=progress)
    wafers = build_wafer_index_map(report['wafer_values'])
    codes, weights, positions = build_history_codes(data, min_coverage=HISTORY_MIN_COVERAGE)
    if progress:
        progress(f'이력 유사성: {codes.shape[1]:,}개 공정 위치 / membership {membership.nnz:,}개')
    model = fit_history_covariance(
        codes, report['wafer_values']['analysis_y'].to_numpy(), weights=weights,
        max_landmarks=HISTORY_MAX_LANDMARKS, max_fit=HISTORY_FIT_WAFERS,
        seed=HISTORY_SEED, enabled=HISTORY_CORRECTION and len(points) > 0, progress=progress)
    if not model['diagnostics'].get('is_available', False):
        # Keep the original result available if a history model cannot be used.
        for name in ('variance_mean', 'variance_contrast', 'n_eff', 'se', 'z',
                     'c', 'c_mc_se', 'lower', 'upper'):
            points[f'history_{name}'] = np.nan
        points['history_mc_draws'] = 0
        points['history_outside'] = pd.Series(pd.NA, index=points.index, dtype='boolean')
        points['history_direction'] = '보정 없음'
        points['history_comparison'] = '보정 없음'
        summary = report['envelope'].copy()
        for name in ('history_c', 'history_c_mc_se'):
            summary[name] = np.nan
        summary['history_mc_draws'] = 0
        report.update({'points': points, 'history_envelope': summary,
                       'membership': membership, 'wafer_index': wafers,
                       'history_positions': positions, 'history_fit': model['diagnostics'],
                       'history_fit_candidates': model['fit_table'],
                       'history_mc': {'status': 'unavailable', 'draws_requested': HISTORY_MC_DRAWS,
                                      'draws_used': 0, 'message': model['diagnostics']['message']},
                       'membership_info': membership_info})
        if progress:
            progress('이력 보정 산출 불가: ' + model['diagnostics']['message'])
        return report
    metrics = covariance_metrics(model, membership)
    for name in metrics:
        points[f'history_{name}'] = metrics[name].to_numpy()
    points['history_se'] = np.sqrt(points['history_variance_contrast'])
    delta = points['sample_mean'].to_numpy(float) - report['mu_hat']
    se = points['history_se'].to_numpy(float)
    valid = np.sqrt(points['history_variance_contrast'].to_numpy(float) / model['sigma2']) > 1e-12
    points['history_z'] = np.divide(delta, se, out=np.zeros_like(delta), where=valid)
    # A zero-variance contrast must also be identically zero under this model.
    impossible = ~valid & (np.abs(delta) > 1e-9 * max(report['sigma_hat'], 1e-100))
    points.loc[impossible, 'history_z'] = np.sign(delta[impossible]) * np.inf
    coeff, mc_info = history_expected_extremes(
        model, membership, points['n'].to_numpy(), points['history_variance_contrast'].to_numpy(),
        draws=HISTORY_MC_DRAWS, min_draws=min(128, HISTORY_MC_DRAWS),
        operation_budget=HISTORY_MC_OPERATION_BUDGET, seed=HISTORY_SEED + 1, progress=progress)
    mc_info['zero_variance_nonzero_contrast_points'] = int(impossible.sum())
    lookup = coeff.set_index('n')
    for name in ('history_c', 'history_c_mc_se', 'history_mc_draws'):
        points[name] = points['n'].map(lookup[name])
    points['history_lower'] = report['mu_hat'] - points['history_se'] * points['history_c']
    points['history_upper'] = report['mu_hat'] + points['history_se'] * points['history_c']
    tolerance = 64 * np.finfo(float).eps * np.maximum(1.0, points['history_c'].to_numpy(float))
    points['history_outside'] = (np.abs(points['history_z']) > points['history_c'] + tolerance).astype('boolean')
    points['history_direction'] = np.select(
        [points['history_z'] > points['history_c'] + tolerance,
         points['history_z'] < -points['history_c'] - tolerance], ['위쪽', '아래쪽'], default='경계 안')
    original = points['outside'].fillna(False).to_numpy(bool)
    corrected = points['history_outside'].fillna(False).to_numpy(bool)
    points['history_comparison'] = np.select(
        [original & corrected, original, corrected],
        ['양쪽 경계 밖', '기존만 경계 밖', '이력 보정만 경계 밖'], default='양쪽 경계 안')
    summary = report['envelope'].merge(coeff, on=['n', 'm_n'], how='left', validate='one_to_one')
    report.update({'points': points, 'history_envelope': summary,
                   'membership': membership, 'wafer_index': wafers,
                   'history_positions': positions,
                   'history_fit': model['diagnostics'], 'history_fit_candidates': model['fit_table'],
                   'history_mc': mc_info, 'membership_info': membership_info})
    if KEEP_HISTORY_MODEL:
        report['history_model'] = model
    if progress:
        progress(f"이력 모델: {model['diagnostics'].get('status')} / Gaussian 기대 극값 {mc_info['draws_used']:,}회")
    return report


def combination_wafers(report, combo_key):
    """Inspect one point's actual wafers, raw y, and analysis y without recomputing."""
    matches = np.flatnonzero(report['points']['combo_key'].to_numpy() == combo_key)
    if len(matches) != 1:
        raise KeyError(f'Expected one combination for {combo_key!r}; found {len(matches)}')
    row = report['membership'].getrow(int(matches[0]))
    result = report['wafer_index'].iloc[row.indices].copy()
    values = report['wafer_values'].iloc[row.indices]
    result['raw_y'] = values['raw_y'].to_numpy()
    result['analysis_y'] = values['analysis_y'].to_numpy()
    return result.reset_index(drop=True)


def analyze_part(part_raw, part_id, *, transform=None, use_yeo_johnson=None, lambda_method=None):
    start = time.perf_counter()
    report = analyze_part_baseline(part_raw, part_id, transform=transform,
                                   use_yeo_johnson=use_yeo_johnson, lambda_method=lambda_method)
    data, reference = report.pop('_history_data'), report.pop('_history_reference')
    report = attach_history_result(report, data, reference)
    report['seconds'] = time.perf_counter() - start
    return report


In [ ]:
results_by_part = {}
rows = []
# 제품마다 전체 CSV를 다시 스캔하지 않도록 한 번에 분할한다.
part_groups = raw.partition_by('part_id', as_dict=True)
for part_id in selected_parts:
    if SHOW_PROGRESS:
        print(f'part {part_id} 분석 시작', flush=True)
    try:
        report = analyze_part(part_groups.pop((part_id,)), part_id)
    except ValueError as exc:
        print(f'  분석 제외: {exc}', flush=True)
        rows.append({'part_id': part_id, '상태': str(exc)})
        continue
    results_by_part[part_id] = report
    points, quality = report['points'], report['quality']
    raw_out = points['raw_outside'].fillna(False).to_numpy(bool)
    active_out = points['outside'].fillna(False).to_numpy(bool)
    rows.append({'part_id': part_id, '상태': '완료', '분석 웨이퍼': len(report['wafer_values']),
                 'G/B 외 제외 웨이퍼': quality['excluded_non_gb'], 'raw y 평균': report['mu_hat_raw'],
                 'raw y 표준편차': report['sigma_hat_raw'],
                 '극값 계수': EXTREME_COEFFICIENT, '변환': report['transform'], 'lambda': report['transform_lambda'],
                 '변환 처리': report['transform_fit']['used'], '변환 상태': report['transform_fit']['status'],
                 '변환 참고': report['transform_fit']['note'],
                 '최대 동률 비율': report['transform_fit'].get('max_tie_fraction', np.nan),
                 '분석값 평균': report['mu_hat'], '분석값 표준편차': report['sigma_hat'],
                 '깊이별 후보': str(report['depth_count'][1:]),
                 '표시 타점 수': len(points), '경계 밖': int(points['outside'].fillna(False).sum()),
                 'raw 경계 밖': int(raw_out.sum()), '양쪽 경계 밖': int((raw_out & active_out).sum()),
                 'raw만 경계 밖': int((raw_out & ~active_out).sum()),
                 '분석값만 경계 밖': int((~raw_out & active_out).sum()),
                 '보정 상태': report['extreme_fit']['status'], '보정 참고': report['extreme_fit']['note'],
                 'κ': report['extreme_fit']['kappa'], 'sigma_eff': report['extreme_fit']['sigma_effective'],
                 '적합 n 구간': report['extreme_fit']['n_groups_used'],
                 '보정 경계 밖': int(points['empirical_outside'].sum()) if report['extreme_fit']['status'] == 'ok' else pd.NA,
                 '기존만 경계 밖': int(points['envelope_comparison'].eq('기존만 경계 밖').sum()) if report['extreme_fit']['status'] == 'ok' else pd.NA,
                 '보정만 경계 밖': int(points['envelope_comparison'].eq('보정만 경계 밖').sum()) if report['extreme_fit']['status'] == 'ok' else pd.NA,
                 '이력 모형 상태': report['history_fit'].get('status'),
                 '이력 경계 밖': int(points['history_outside'].sum()) if report['history_fit'].get('is_available') else pd.NA,
                 '이력 rho': report['history_fit'].get('rho'), '이력 ell': report['history_fit'].get('ell'),
                 '기대 극값 MC 횟수': report['history_mc']['draws_used'],
                 '경계 없음': int(points['outside'].isna().sum()), '계산 초': report['seconds']})
    print(f"  완료: raw 평균 {report['mu_hat_raw']:.10g} · 분석 평균 {report['mu_hat']:.10g} · 깊이별 후보 {report['depth_count'][1:]} · 표시 {len(points):,}개 · {report['seconds']:.2f}초", flush=True)
part_summary = pd.DataFrame(rows).set_index('part_id')
display(part_summary)


## 3. 변환 전·후 웨이퍼 y 분포
Part별 **고유 웨이퍼**의 raw y와 분석값을 비교합니다. 위 행은 히스토그램과 같은 평균·표준편차의 정규 밀도, 아래 행은 정규 Q–Q plot입니다. OFF이면 두 열 모두 raw y입니다.

히스토그램은 극단값을 자르지 않고 전체 범위를 표시합니다. Q–Q plot은 양쪽 꼬리를 포함하며, 웨이퍼가 5,000장을 넘으면 표시 점만 순위별로 균등하게 줄입니다. 순위 계산·λ 추정과 모든 통계는 전체 웨이퍼를 사용합니다. 정규 Q–Q 기준선은 전체 평균·표준편차로 그립니다. 정규성이 좋아졌는지, 꼬리의 큰 값이 얼마나 압축됐는지 함께 확인하세요.

`DISTRIBUTION_PART_IDS = None`이면 모든 part, 예를 들어 `['PRODUCT_A']`이면 해당 part만 표시합니다. 각 단위에서 정확히 0인 값의 비율과 최대 동률 비율도 표에서 확인할 수 있습니다. Raw의 0은 순위변환 후 일반적으로 0이 아닌 동일 점수로 바뀝니다.


In [ ]:
DISTRIBUTION_PART_IDS = None  # None: 모든 part. 예: ['PRODUCT_A']


def plot_y_distributions(report):
    from statistics import NormalDist
    wf = report['wafer_values']
    raw_y = wf['raw_y'].to_numpy(dtype=float)
    analysis_y = wf['analysis_y'].to_numpy(dtype=float)
    enabled = report['transform'] != 'raw'
    after_label = 'After: ' + analysis_value_label(report)
    fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
    n = len(raw_y)
    ranks = np.unique(np.linspace(0, n - 1, min(n, 5000), dtype=int))
    normal = NormalDist()
    theoretical = np.array([normal.inv_cdf((int(i) + 0.5) / n) for i in ranks])
    diagnostics = []
    for col, (label, values) in enumerate([('Before: raw y', raw_y), (after_label, analysis_y)]):
        mu = float(np.mean(values))
        sd = float(np.std(values, ddof=1))
        vmin, vmax = float(values.min()), float(values.max())
        # 저카운트 정수는 정수마다 한 bin, 그 외에는 유한한 수의 균등 bin을 사용한다.
        if vmax - vmin <= 60 and np.equal(values, np.round(values)).all():
            bins = np.arange(vmin - 0.5, vmax + 1.5, 1.0)
        else:
            bins = min(60, max(10, int(np.sqrt(n))))
        ax = axes[0, col]
        ax.hist(values, bins=bins, density=True, alpha=0.6, color='#3878ad', label='Wafer values')
        xs = np.linspace(vmin, vmax, 400)
        density = np.exp(-0.5 * ((xs - mu) / sd)**2) / (sd * np.sqrt(2 * np.pi))
        ax.plot(xs, density, color='#d16a35', linewidth=1.5, label='Normal: same mean / SD')
        ax.set(title=label, xlabel='y (raw)' if col == 0 or not enabled else 'T(y)', ylabel='Density')
        ax.legend(fontsize=8)
        qq = axes[1, col]
        ordered = np.sort(values)
        qq.scatter(theoretical, ordered[ranks], s=SCATTER_SIZE, marker='.', linewidths=0,
                   alpha=0.45, color='#3878ad', rasterized=True)
        qq.plot(theoretical, mu + sd * theoretical, color='#d16a35', linewidth=1.5)
        qq.set(title='Normal Q–Q plot', xlabel='Theoretical standard-normal quantile',
               ylabel='Observed raw y' if col == 0 or not enabled else 'Observed T(y)')
        for a in (ax, qq):
            a.grid(alpha=0.2)
            a.ticklabel_format(axis='both', style='plain', useOffset=False)
        diagnostics.append({'단위': label, '웨이퍼 수': n, '평균': mu, '표준편차': sd,
                            '왜도': float(pd.Series(values).skew()),
                            '0 비율': float(np.mean(values == 0)),
                            '최대 동률 비율': float(np.unique(values, return_counts=True)[1].max() / n),
                            '최솟값': vmin, '최댓값': vmax})
    mode = analysis_mode_label(report)
    fig.suptitle(f"part_id = {report['part_id']}  |  {mode}  |  K={n:,}")
    plt.show()
    display(pd.DataFrame(diagnostics).set_index('단위'))
    return fig


distribution_parts = (list(results_by_part) if DISTRIBUTION_PART_IDS is None
                      else ([str(DISTRIBUTION_PART_IDS)] if isinstance(DISTRIBUTION_PART_IDS, (str, int))
                            else list(map(str, DISTRIBUTION_PART_IDS))))
for part_id in distribution_parts:
    if part_id not in results_by_part:
        print(f'part {part_id}: 완료된 분석 결과가 없습니다.')
        continue
    part_heading(part_id)
    plot_y_distributions(results_by_part[part_id])


## 4. 기존 경계와 이력·중복 보정 경계

왼쪽 y축은 실제 조합의 분석값 평균, 오른쪽은 `(조합 평균 - part 평균) / 보정 SE`입니다.
같은 n에서도 구성에 따라 SE가 달라지므로 보정 후에는 표준화해서 한 경계를 표시합니다.
아래의 n_eff/n은 평균 자체의 정보량 진단으로, 오른쪽 점수의 분모와는 다릅니다.
곡선의 높이는 각 n의 실제 m_n과 타점 사이의 공동 분포를 반영합니다.
색은 두 주 경계의 판정만 비교하며, 선택적으로 표시한 κ 참고선은 색 판정에 사용하지 않습니다.


In [ ]:
def plot_part(report):
    points, summary = report['points'], report['envelope']
    part_id, mu_hat = report['part_id'], report['mu_hat']
    fit = report['extreme_fit']
    calibrated = fit['status'] == 'ok'
    if points.empty:
        print(f'part {part_id}: MIN_N / MAX_DEPTH / STEP 조건에 맞는 관측 조합이 없습니다.')
        return None
    fig, (ax, axm, axs) = plt.subplots(3, 1, figsize=(14, 9), sharex=True,
                                 gridspec_kw={'height_ratios': [4, 1, 1], 'hspace': 0.10})
    # 판정 집합을 서로 겹치지 않게 나누고, 모든 점에 같은 크기/모양을 적용한다.
    original = points['outside'].fillna(False).to_numpy(bool)
    corrected = points['empirical_outside'].fillna(False).to_numpy(bool)
    if calibrated:
        groups = [
            (~(original | corrected), '#65758b', '두 경계 안'),
            (original & ~corrected, '#2876bd', '기본 경계만 밖'),
            (~original & corrected, '#24846b', 'σ 보정 경계만 밖'),
            (original & corrected, '#d44838', '두 경계 모두 밖'),
        ]
    else:
        groups = [(~original, '#65758b', '기본 경계 안'),
                  (original, '#2876bd', '기본 경계 밖')]
    for mask, color, label in groups:
        if not mask.any():
            continue
        ax.scatter(points.loc[mask, 'n'], points.loc[mask, 'sample_mean'],
                   s=SCATTER_SIZE, marker='.', linewidths=0, alpha=0.5,
                   color=color, rasterized=True, label=f'{label} {mask.sum():,}개')
    x = summary['n'].to_numpy(float)
    ax.plot(x, summary['upper'], '--', color='#2876bd', linewidth=1.4, label='기본 envelope')
    ax.plot(x, summary['lower'], '--', color='#2876bd', linewidth=1.4)
    if calibrated:
        ax.plot(x, summary['empirical_upper'], '-', color='#24846b', linewidth=1.5,
                label=rf'σ 보정 envelope · $\kappa={fit["kappa"]:.3g}$')
        ax.plot(x, summary['empirical_lower'], '-', color='#24846b', linewidth=1.5)
    ax.axhline(mu_hat, color='#202b3b', linestyle=':', linewidth=1.2,
               label=f'part 평균 = {mu_hat:.5g}')
    coefficient_label = ('유한 m: E[max|Z|]' if summary['coefficient_method'].iloc[0] == 'absolute_max'
                         else '이전 근사: sqrt(2 log m)')
    detail = (f"sigma0={report['sigma_hat']:.4g} → sigma_eff={fit['sigma_effective']:.4g} · "
              f"적합 n 구간 {fit['n_groups_used']:,}개" if calibrated
              else f"경험적 보정: {fit['status']} · {fit['note']}")
    ax.set_title(f"part_id = {part_id}  |  {analysis_mode_label(report)} · {coefficient_label}\n{detail}", loc='left', pad=12)
    ax.set_ylabel(f'조합의 {analysis_value_label(report)} 평균')
    ax.ticklabel_format(axis='y', style='plain', useOffset=False)
    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), frameon=False, fontsize=9)
    axm.vlines(x, 0, summary['m_n'].to_numpy(), color='#344964', linewidth=1)
    axm.scatter(x, summary['m_n'], color='#344964', s=SCATTER_SIZE, marker='.', linewidths=0)
    axm.set_ylabel('타점 수 m_n')
    axm.set_ylim(bottom=0)
    axs.plot(x, summary['sigma_hat_n'], '--', color='#2876bd', linewidth=1.2, label='기존 sigma0 / √n')
    if calibrated:
        axs.plot(x, summary['empirical_sigma_hat_n'], '-', color='#24846b', linewidth=1.3,
                 label='보정 sigma_eff / √n')
    axs.set(xlabel='조합의 고유 웨이퍼 수 n', ylabel='평균의 척도')
    axs.legend(loc='upper left', bbox_to_anchor=(1.01, 1), frameon=False, fontsize=9)
    if LOG_X:
        axm.set_xscale('log')
        axm.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{value:g}'))
    for a in (ax, axm, axs):
        a.grid(alpha=0.2)
        a.spines[['top', 'right']].set_visible(False)
    fig.subplots_adjust(right=0.73, left=0.10, top=0.88, bottom=0.09)
    plt.show()
    return fig


def plot_extreme_fit(report):
    """표시 envelope가 아니라, σ 적합에 쓴 정확 기대 극값과 잔차를 확인한다."""
    fit = report['extreme_fit']
    if fit['status'] != 'ok':
        return None
    d = report['extreme_fit_diagnostics']
    d = d.loc[d['fit_used']].sort_values('n')
    x = d['n'].to_numpy(float)
    fig, (ax, ar) = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                                gridspec_kw={'height_ratios': [3, 2], 'hspace': 0.12})
    ax.scatter(x, d['d_plus'], s=SCATTER_SIZE, marker='.', linewidths=0, color='#c15a39', alpha=0.65, label='관찰 상단: √n (max - μ) / sigma0')
    ax.scatter(x, d['d_minus'], s=SCATTER_SIZE, marker='.', linewidths=0, color='#765eac', alpha=0.6,
               label='관찰 하단: √n (μ - min) / sigma0')
    ax.plot(x, d['a_m'], '--', color='#2876bd', linewidth=1.2, label='기존 기대 극값 a_m')
    ax.plot(x, fit['kappa'] * d['a_m'], '-', color='#24846b', linewidth=1.5,
            label='적합 기대 극값 κ × a_m')
    ax.set_ylabel('표준화한 극값 거리')
    ax.set_title(f"part_id = {report['part_id']}  |  극값 적합 진단 · κ={fit['kappa']:.4g}", loc='left')
    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=8, frameon=False)
    ar.scatter(x, d['fit_resid_plus'], s=SCATTER_SIZE, marker='.', linewidths=0, color='#c15a39', alpha=0.65, label='상단 잔차')
    ar.scatter(x, d['fit_resid_minus'], s=SCATTER_SIZE, marker='.', linewidths=0, color='#765eac', alpha=0.6, label='하단 잔차')
    ar.axhline(0, color='#293342', linewidth=1)
    for level in (-EXTREME_FIT_HUBER_DELTA, EXTREME_FIT_HUBER_DELTA):
        ar.axhline(level, color='#9b9b9b', linestyle=':', linewidth=0.8)
    ar.set(xlabel='조합의 고유 웨이퍼 수 n', ylabel='적합 잔차 / 정규 극값 SD')
    ar.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=8, frameon=False)
    if LOG_X:
        ar.set_xscale('log')
        ar.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{value:g}'))
    for a in (ax, ar):
        a.grid(alpha=0.2)
        a.spines[['top', 'right']].set_visible(False)
    fig.subplots_adjust(right=0.68, left=0.10, top=0.92, bottom=0.10)
    plt.show()
    return fig


def plot_history_part(report):
    """Original mean units on the left; centered covariance scores on the right."""
    points = report['points']
    if points.empty:
        print(f"part {report['part_id']}: 표시할 공정조합이 없습니다.")
        return None
    original = points['outside'].fillna(False).to_numpy(bool)
    corrected = points['history_outside'].fillna(False).to_numpy(bool)
    available = report['history_fit'].get('is_available', False)
    groups = [
        (~(original | corrected), '#65758b', '두 경계 안'),
        (original & ~corrected, '#2876bd', '기존만 밖'),
        (~original & corrected, '#24846b', '이력 보정만 밖'),
        (original & corrected, '#d44838', '두 경계 모두 밖'),
    ]
    if not available:
        groups = [(~original, '#65758b', '기존 경계 안'),
                  (original, '#2876bd', '기존 경계 밖')]
    fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharex='col',
                            gridspec_kw={'height_ratios': [4, 1], 'hspace': 0.10, 'wspace': 0.20})
    left, right = axes[0]
    for mask, color, label in groups:
        if not mask.any():
            continue
        for ax, field in ((left, 'sample_mean'), (right, 'history_z')):
            if field == 'history_z' and not available:
                continue
            ax.scatter(points.loc[mask, 'n'], points.loc[mask, field],
                       s=SCATTER_SIZE, marker='.', linewidths=0, alpha=0.5,
                       color=color, rasterized=True, label=f'{label} {mask.sum():,}')
    base, corrected_curve = report['envelope'], report['history_envelope']
    left.plot(base['n'], base['upper'], '--', color='#2876bd', lw=1.4, label='기존 독립 envelope')
    left.plot(base['n'], base['lower'], '--', color='#2876bd', lw=1.4)
    if SHOW_LEGACY_SCALE_FIT and report['extreme_fit']['status'] == 'ok':
        left.plot(base['n'], base['empirical_upper'], ':', color='#8d65aa', lw=1.1, label='참고: 기존 κ 적합')
        left.plot(base['n'], base['empirical_lower'], ':', color='#8d65aa', lw=1.1)
    left.axhline(report['mu_hat'], color='#202b3b', linestyle=':', lw=1)
    left.set(title='기존 평균 차트', ylabel=f"조합의 {analysis_value_label(report)} 평균")
    left.ticklabel_format(axis='y', style='plain', useOffset=False)
    if available:
        right.plot(corrected_curve['n'], corrected_curve['history_c'], '-', color='#24846b', lw=1.5,
                   label='공동 Gaussian E[max|Z|]')
        right.plot(corrected_curve['n'], -corrected_curve['history_c'], '-', color='#24846b', lw=1.5)
    if SHOW_IID_COEFFICIENT and available:
        iid_c = corrected_curve['m_n'].map(expected_normal_absolute_max)
        right.plot(corrected_curve['n'], iid_c, '--', color='#9ca3af', lw=1.0, label='참고: 독립 타점 E[max|Z|]')
        right.plot(corrected_curve['n'], -iid_c, '--', color='#9ca3af', lw=1.0)
    right.axhline(0, color='#202b3b', linestyle=':', lw=1)
    right.set(title='이력·웨이퍼 중복·전체 평균을 반영한 비교',
              ylabel='(조합 평균 - part 평균) / 보정 SE')
    if not report['history_fit'].get('is_available', False):
        right.text(0.5, 0.5, '보정 산출 불가 — 아래 적합 상태 확인', transform=right.transAxes,
                   ha='center', va='center', color='#b23b32')
    infinite_count = int(np.isinf(points['history_z'].to_numpy(float)).sum())
    if infinite_count:
        right.text(0.01, 0.02, f'모형 분산 0과 불일치한 ∞ 점수 {infinite_count}개: 아래 표 확인',
                   transform=right.transAxes, color='#b23b32', fontsize=8)
    axes[1, 0].vlines(base['n'], 0, base['m_n'], color='#566475', lw=1)
    axes[1, 0].set(ylabel='타점 수 m_n', xlabel='조합의 고유 웨이퍼 수 n', ylim=(0, None))
    axes[1, 1].scatter(points['n'], points['history_n_eff'] / points['n'],
                       s=SCATTER_SIZE, marker='.', linewidths=0, alpha=0.25,
                       color='#566475', rasterized=True)
    neff_ratio_max = float((points['history_n_eff'] / points['n']).max())
    axes[1, 1].set(ylabel='진단: n_eff / n', xlabel='조합의 고유 웨이퍼 수 n',
                   ylim=(0, max(1.05, neff_ratio_max * 1.05) if np.isfinite(neff_ratio_max) else 1.05))
    for ax in axes.flat:
        if LOG_X:
            ax.set_xscale('log')
            ax.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{value:g}'))
            # Also valid when all corrected y values are unavailable (NaN).
            ax.set_xlim(float(points['n'].min()) / 1.08, float(points['n'].max()) * 1.08)
        ax.grid(alpha=0.18)
        ax.spines[['top', 'right']].set_visible(False)
    for ax in (left, right):
        if ax.get_legend_handles_labels()[0]:
            ax.legend(loc='upper right', fontsize=8, frameon=True, framealpha=0.9)
    fit = report['history_fit']
    rho = fit.get('rho', np.nan)
    ell = fit.get('ell', np.nan)
    method = ('finite E[max|Z|]' if base['coefficient_method'].iloc[0] == 'absolute_max' else 'legacy sqrt(2 log m)')
    fig.suptitle(f"part_id = {report['part_id']}  |  {analysis_mode_label(report)}\n"
                 f"기존: {method} · 이력 모델 {fit.get('status', '')} · ρ={rho:.3g}, ℓ={ell:.3g} · "
                 f"MC {report['history_mc']['draws_used']:,}회", fontsize=13)
    fig.subplots_adjust(left=0.07, right=0.98, bottom=0.08, top=0.87)
    plt.show()
    return fig


for part_id, report in results_by_part.items():
    part_heading(part_id)
    plot_history_part(report)


## 5. 공분산 적합·수치 오차·경계 밖 조합 확인

`history_fit_candidates`는 독립 모형까지 포함한 REML 후보 비교입니다.
`history_fit`의 상태, 근사 오차, 이력 coverage 및 MCSE를 함께 확인하세요.
`history_se`는 전체 part 평균과의 차이의 표준오차이며,
`history_lower/upper`는 점마다 다른 원래 분석값 단위 경계입니다.

특정 타점의 웨이퍼 목록:
```python
report = results_by_part['PART_ID']
combination_wafers(report, report['points'].iloc[0]['combo_key'])
```
`membership`의 행은 `points` 순서, 열은 `wafer_index` 순서입니다. CSR 형식으로
저장하므로 조합마다 긴 wafer ID 문자열을 반복 저장하지 않습니다.


In [ ]:
for part_id, report in results_by_part.items():
    part_heading(part_id)
    print('공분산 적합 상태:')
    display(pd.DataFrame([report['history_fit']]).T.rename(columns={0: '값'}))
    print('공분산 후보 비교:')
    display(report['history_fit_candidates'])
    print('모형의 기대 최대 절대편차 계산:')
    display(pd.DataFrame([report['history_mc']]).T.rename(columns={0: '값'}))
    display(report['history_positions'].groupby(['kept', 'reason']).size().rename('공정 위치 수').to_frame())
    display(report['history_envelope'][['n', 'm_n', 'extreme_coefficient', 'history_c',
                                       'history_c_mc_se', 'history_mc_draws']])
    display(report['points']['history_comparison'].value_counts().rename('조합 수').to_frame())
    points = report['points']
    review = points.loc[points['outside'].fillna(False) | points['history_outside'].fillna(False)].copy()
    review = review.sort_values('history_z', key=lambda x: x.abs(), ascending=False)
    display(review[['combo_key', 'step', 'depth', 'path', 'n', 'm_n', 'raw_mean', 'sample_mean',
                    'history_n_eff', 'history_se', 'history_z', 'history_c',
                    'history_lower', 'history_upper', 'history_comparison']].head(OUTSIDE_ROWS))


## 선택: 저장 및 타점 구성 웨이퍼 확인

`SAVE_DIR`를 지정하면 기존 결과, 이력 경계·적합 진단·웨이퍼 목록을 저장합니다.
membership NPZ 파일의 행은 `means.csv` 행 순서, 열은 `wafer_index.csv`의 `wafer_index`입니다.
`wafer_index.csv`에는 raw y와 분석값도 저장합니다.


In [ ]:
if SAVE_DIR is not None:
    save_dir = Path(SAVE_DIR).expanduser()
    save_dir.mkdir(parents=True, exist_ok=True)
    part_summary.to_csv(save_dir / 'part_summary.csv', encoding='utf-8-sig')
    for part_id, report in results_by_part.items():
        safe = re.sub(r'[^\w.-]+', '_', str(part_id))[:80] or 'part'
        suffix = hashlib.sha256(str(part_id).encode()).hexdigest()[:8]
        stem = f'{safe}_{suffix}'
        report['points'].to_csv(save_dir / f'{stem}_means.csv', index=False, encoding='utf-8-sig')
        report['envelope'].to_csv(save_dir / f'{stem}_envelope.csv', index=False, encoding='utf-8-sig')
        pd.DataFrame([report['extreme_fit']]).to_csv(save_dir / f'{stem}_extreme_fit.csv', index=False, encoding='utf-8-sig')
        report['extreme_fit_diagnostics'].to_csv(save_dir / f'{stem}_extreme_diagnostics.csv', index=False, encoding='utf-8-sig')
        report['raw_envelope'].to_csv(save_dir / f'{stem}_raw_envelope.csv', index=False, encoding='utf-8-sig')
        report['history_envelope'].to_csv(save_dir / f'{stem}_history_envelope.csv', index=False, encoding='utf-8-sig')
        report['history_fit_candidates'].to_csv(save_dir / f'{stem}_history_fit_candidates.csv', index=False, encoding='utf-8-sig')
        pd.DataFrame([report['history_fit']]).to_csv(save_dir / f'{stem}_history_fit.csv', index=False, encoding='utf-8-sig')
        pd.DataFrame([report['history_mc']]).to_csv(save_dir / f'{stem}_history_mc.csv', index=False, encoding='utf-8-sig')
        report['history_positions'].to_csv(save_dir / f'{stem}_history_positions.csv', index=False, encoding='utf-8-sig')
        wafer_export = report['wafer_index'].copy()
        wafer_export['raw_y'] = report['wafer_values']['raw_y'].to_numpy()
        wafer_export['analysis_y'] = report['wafer_values']['analysis_y'].to_numpy()
        wafer_export.to_csv(save_dir / f'{stem}_wafer_index.csv', index=False, encoding='utf-8-sig')
        sparse.save_npz(save_dir / f'{stem}_membership.npz', report['membership'])
    print(f'저장 위치: {save_dir.resolve()}')

PART_TO_VIEW = None  # 예: 'PRODUCT_A'
if PART_TO_VIEW is not None:
    plot_history_part(results_by_part[str(PART_TO_VIEW)])

# 타점 구성 예시:
# report = results_by_part['PRODUCT_A']
# display(combination_wafers(report, report['points'].iloc[0]['combo_key']))
